In [1]:
# Réinstallation de numpy (build incohérent) si besoin, puis Restart runtime
%pip install --force-reinstall --no-cache-dir "numpy>=2.0"
%pip install permetrics
%pip install mapie
%pip install lightgbm
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 231.9 MB/s eta 0:00:0000:010:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.2
    Uninstalling numpy-2.5.2:
      Successfully uninstalled numpy-2.5.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.2 which is incompatible.


KeyboardInterrupt: 

In [2]:
# @title Import packages
import os
import json
import random
import time
from datetime import datetime, timedelta
from typing import List, Tuple

import numpy as np
import numpy._core._multiarray_umath as _mu
if not hasattr(_mu, '_blas_supports_fpe'):
    raise RuntimeError(
        f"numpy incohérent (v{np.__version__}, fonction _blas_supports_fpe absente du module C : "
        f"build incomplet). Exécuter la cellule 0 (réinstallation), puis Runtime -> Restart runtime.")
print('numpy', np.__version__, 'OK')
import pandas as pd

import lightgbm as lgb
from mapie.regression import SplitConformalRegressor, ConformalizedQuantileRegressor
from permetrics import RegressionMetric

FOLDER_NAME = "station_depth_csv"

from google.colab import drive
drive.mount('/content/gdrive')
os.chdir("/content/gdrive/My Drive/Soil_Moisture/Local_Training")
ROOT_DIR = "/content/gdrive/My Drive/Soil_Moisture/dataset_training"
drive_dir = os.path.join("/content/gdrive/My Drive/Soil_Moisture/outputs", "Fine_Tuning_Osiris")
    

RESULTS_CSV_PATH = os.path.join(drive_dir, "results.csv")

Mounted at /content/gdrive


In [3]:
# ============================================================
# Configuration
# ============================================================


MASTER_CSV_PATH = os.path.join(ROOT_DIR, FOLDER_NAME, "Soil_Properties_Master.csv")

base_path = os.path.join(ROOT_DIR, FOLDER_NAME, "depth")

FULL_DENSE_h = [ "soil_moisture", "ET0", "IRRAD", "TMIN", "TMAX", "VAP", "WIND", "RAIN",
              "VPD", "T_RANGE",
              "RAIN_CUM_3D", "RAIN_CUM_7D", "RAIN_CUM_14D",
              "doy_sin", "doy_cos"]

FULL_DENSE = [  "ET0", "IRRAD", "TMIN", "TMAX", "VAP", "WIND", "RAIN",
              "VPD", "T_RANGE",
              "RAIN_CUM_3D", "RAIN_CUM_7D", "RAIN_CUM_14D",
              "doy_sin", "doy_cos"]

FULL_SOIL  = [ "clay", "silt", "bulk", "sand", "dem", "ksat_m_1km", "dem_slope", "dem_aspect", "dem_twi"]

FULL_SPARSE = ["S2_B2", "S2_B3", "S2_B4", "S2_B5", "S2_B6", "S2_B7",
               "S2_B8", "S2_B8A", "S2_B11", "S2_B12",
               "S2_NDVI", "S2_NDWI", "S2_SAVI", "S2_MNDWI", "S2_NBR",
               "S1_VV", "S1_VH", "S1_angle",
               "S1_VV_over_VH", "S1_VH_over_VV"]

FULL_SPARSE_S1 = ["S1_VV", "S1_VH", "S1_angle",
                   "S1_VV_over_VH", "S1_VH_over_VV"]

FULL_SPARSE_S2 = ["S2_B2", "S2_B3", "S2_B4", "S2_B5", "S2_B6", "S2_B7",
               "S2_B8", "S2_B8A", "S2_B11", "S2_B12",
               "S2_NDVI", "S2_NDWI", "S2_SAVI", "S2_MNDWI", "S2_NBR"]

FULL_SPARSE_HLS30 = ["HLS_B2", "HLS_B3", "HLS_B4", "HLS_B5", 
                     "HLS_B6", "HLS_B7", "HLS_B9", "HLS_B10", "HLS_B11", "HLS_NDVI"]

def _without(lst, *items):
    return [x for x in lst if x not in items]

FEATURE_CONFIGS = [
    # {"name": "test", "dense": FULL_DENSE, "soil": FULL_SOIL,  "sparse": FULL_SPARSE},
    {"name": "nb_windows_S2", "dense": FULL_DENSE, "soil": FULL_SOIL,  "sparse": FULL_SPARSE_S2},
]

# La config active est la 1ere entree non commentee de FEATURE_CONFIGS.
# Pour changer d'experience, decommente une autre ligne au-dessus et laisse une seule
# entree active (ou change l'index ci-dessous).
ACTIVE_CONFIG = FEATURE_CONFIGS[0]
DENSE_FEATURE_COLS = ACTIVE_CONFIG["dense"]
SOIL_PROPERTIES_COLS = ACTIVE_CONFIG["soil"]
SPARSE_FEATURE_COLS = ACTIVE_CONFIG["sparse"]
FEATURE_COLS = DENSE_FEATURE_COLS + SOIL_PROPERTIES_COLS + SPARSE_FEATURE_COLS

SAVE_PLOTS = False
SAVE_NETWORKS_DIR = True
SAVE_MODELS_DIR = True
SAVE_RESULTS_CSV = False
TARGET_COL = "soil_moisture"
DATE_COL = "date_time"  # L'index temporel est sauvegardé sous date_time par process_timeseries

# Training settings
HORIZONS = [7]              # predict * days ahead 
LOOKBACK = [1]                  # use past * days to predict next day
EPOCHS = 150
BATCH_SIZE = 32
SEED = 8
DEPTHS = [0.1, 0.5]  # DEPTHS[0]=surface (0.1 m), DEPTHS[1]=rootzone (0.5 m).
# Le pont local (plus bas) utilise DEPTHS[0] pour "surface" et DEPTHS[1] pour "rootzone".
# Avec une seule profondeur, sly et rly pointeront vers les memes donnees.
NETWORKS = [
#     ["COSMOS-UK", "GROW", "PTSMN", "TAHMO", "TERENO"]
# ]
    "all"]
    #  ["COSMOS-UK"],
    #          ["DWD"],
    #          ["FR_Aqui", "GROW"],
    #          ["PTSMN"], 
    #         ["SMOSMANIA"], ["SOILSCAPE"],
    #         ["TAHMO","TERENO"],
    #     ["TERENO"], 
    #         ["TWENTE"], 
    #         ["XMS-CAT"]
    #     ]
NB_WINDOWS = [50000]    
# Models: xgboost and lightgbm are fast, non-DL
MODELS = ["lightgbm"]
# , "lightgbm"]
if SAVE_MODELS_DIR:
    os.makedirs(drive_dir, exist_ok=True)
np.random.seed(SEED)
random.seed(SEED)


In [4]:



def resample_timeseries(df_temp, freq='D', method='mean', start_date=None, end_date=None, specific_hour=None):
    if start_date is not None:
        df_temp = df_temp.loc[start_date:]
    if end_date is not None:
        df_temp = df_temp.loc[:end_date]

    numeric_cols = df_temp.select_dtypes(include='number').columns
    df_temp = df_temp[numeric_cols]

    if freq == 'D' and specific_hour is not None:
        df_temp = df_temp[df_temp.index.hour == specific_hour]
        df_resampled = df_temp.resample('D').first()
    else:
        resampler = df_temp.resample(freq)
        if method == 'mean':
            df_resampled = resampler.mean()
        elif method == 'sum':
            df_resampled = resampler.sum()
        else:
            raise ValueError(f"Méthode '{method}' non reconnue.")
    return df_resampled

def update_soil_property(df, df_master, cols, depth):
    
    df_master = df_master.copy()
    
    lat_local = df['Latitude'].iloc[0]
    lon_local = df['Longitude'].iloc[0]
    
    tol = 1e-4
    match = df_master[
        (np.abs(df_master['latitude'] - lat_local) < tol) & 
        (np.abs(df_master['longitude'] - lon_local) < tol)
    ]
    
    if match.empty:
        print(f" Aucune correspondance trouvée dans le Master CSV pour ({lat_local}, {lon_local})")
        return None

    df = df.rename(columns={'Clay_fraction': 'clay', 'Silt_fraction': 'silt',
                            'Sand_fraction': 'sand', 'Elevation': 'dem'})
    
    idx = match.index[0]                    # (1) index AVANT rename

    if depth <= 0.3:
        df_master = df_master.rename(columns={
            'clay_m_30m_0cm_30cm': 'clay', 'silt_m_30m_0cm_30cm': 'silt',
            'sand_m_30m_0cm_30cm': 'sand', 'bulk_m_30m_0cm_30cm': 'bulk',
            'dem_m_30m_depth': 'dem', 'ksat_m_1km_0cm': 'ksat_m_1km',
        })
    elif depth <= 0.6:
        df_master = df_master.rename(columns={
            'clay_m_30m_30cm_60cm': 'clay', 'silt_m_30m_30cm_60cm': 'silt',
            'sand_m_30m_30cm_60cm': 'sand', 'bulk_m_30m_30cm_60cm': 'bulk',
            'dem_m_30m_depth': 'dem', 'ksat_m_1km_30cm': 'ksat_m_1km',
        })
    elif depth <= 1.0:
        df_master = df_master.rename(columns={
            'clay_m_30m_60cm_100cm': 'clay', 'silt_m_30m_60cm_100cm': 'silt',
            'sand_m_30m_60cm_100cm': 'sand', 'bulk_m_30m_60cm_100cm': 'bulk',
            'dem_m_30m_depth': 'dem', 'ksat_m_1km_60cm': 'ksat_m_1km',
        })
    else:
        return df
    
    row_master = df_master.loc[idx]          # (2) depuis le df renommé

    for c in cols:
        if c not in df.columns:
            if c in row_master.index and pd.notna(row_master[c]):
                df[c] = row_master[c]
            else:
                df[c] = np.nan
        elif c in row_master.index:
            df[c] = df[c].fillna(row_master[c])

    return df

def interpolate_timeseries(df, col='soil_moisture', n=1, method='linear'):
    """
    Interpole les valeurs manquantes d'une série temporelle pour les valeurs isolées.
    Seules les valeurs manquantes entourées de données valides seront interpolées.
    """

    df_interpolated = df.copy()
    df_interpolated[col] = df_interpolated[col].interpolate(method=method, limit=n)
    return df_interpolated

def engineer_features(df):
    df = df.copy()

    # ── doy_sin / doy_cos ──
    if 'doy_sin' in FEATURE_COLS or 'doy_cos' in FEATURE_COLS:
        day_of_year = df[DATE_COL].dt.dayofyear
        df['doy_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
        df['doy_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)

    # ── VPD ──
    if 'VPD' in FEATURE_COLS and all(c in df.columns for c in ['TMAX', 'TMIN', 'VAP']):
        t_mean = (df['TMAX'] + df['TMIN']) / 2
        es = 0.6108 * np.exp(17.27 * t_mean / (t_mean + 237.3))
        df['VPD'] = es - df['VAP']

    # ── T_RANGE ──
    if 'T_RANGE' in FEATURE_COLS and all(c in df.columns for c in ['TMAX', 'TMIN']):
        df['T_RANGE'] = df['TMAX'] - df['TMIN']

    # ── Rain accumulation ──
    for window, col in [(3, 'RAIN_CUM_3D'), (7, 'RAIN_CUM_7D'), (14, 'RAIN_CUM_14D')]:
        if col in FEATURE_COLS and 'RAIN' in df.columns:
            df[col] = df['RAIN'].rolling(window=window, min_periods=1).sum()

    # ── S2 indices ──
    if 'S2_NDWI' in FEATURE_COLS and all(c in df.columns for c in ['S2_B3', 'S2_B8']):
        num = df['S2_B3'] - df['S2_B8']
        denom = df['S2_B3'] + df['S2_B8']
        df['S2_NDWI'] = np.where(denom != 0, num / denom, np.nan)

    if 'S2_SAVI' in FEATURE_COLS and all(c in df.columns for c in ['S2_B4', 'S2_B8']):
        L = 0.5
        num = df['S2_B8'] - df['S2_B4']
        denom = df['S2_B8'] + df['S2_B4'] + L
        df['S2_SAVI'] = np.where(denom != 0, (num / denom) * (1 + L), np.nan)

    if 'S2_MNDWI' in FEATURE_COLS and all(c in df.columns for c in ['S2_B3', 'S2_B11']):
        num = df['S2_B3'] - df['S2_B11']
        denom = df['S2_B3'] + df['S2_B11']
        df['S2_MNDWI'] = np.where(denom != 0, num / denom, np.nan)

    if 'S2_NBR' in FEATURE_COLS and all(c in df.columns for c in ['S2_B8', 'S2_B12']):
        num = df['S2_B8'] - df['S2_B12']
        denom = df['S2_B8'] + df['S2_B12']
        df['S2_NBR'] = np.where(denom != 0, num / denom, np.nan)

    # ── S1 ratios ──
    if all(c in df.columns for c in ['S1_VV', 'S1_VH']):
        if 'S1_VV_over_VH' in FEATURE_COLS:
            df['S1_VV_over_VH'] = np.where(df['S1_VH'] != 0, df['S1_VV'] / df['S1_VH'], np.nan)
        if 'S1_VH_over_VV' in FEATURE_COLS:
            df['S1_VH_over_VV'] = np.where(df['S1_VV'] != 0, df['S1_VH'] / df['S1_VV'], np.nan)



    # ── ET₀ FAO Penman-Monteith ──
    if 'ET0' in FEATURE_COLS and all(c in df.columns for c in ['TMAX', 'TMIN', 'WIND', 'IRRAD']):
        # Altitude
        elev = df['dem'].iloc[0] if 'dem' in df.columns else None
        if elev is None:
            elev = 0
        
        # Constantes
        t_mean = (df['TMAX'] + df['TMIN']) / 2
        
        # Pression atmosphérique (kPa)
        P = 101.3 * ((293 - 0.0065 * elev) / 293) ** 5.26
        
        # Constante psychrométrique (kPa/°C)
        gamma = 0.665e-3 * P
        
        # Pente de la courbe de saturation (kPa/°C)
        es_tmean = 0.6108 * np.exp(17.27 * t_mean / (t_mean + 237.3))
        Delta = 4098 * es_tmean / (t_mean + 237.3) ** 2
        
        # VPD (kPa) — déjà calculé ou à recalculer
        if 'VPD' in df.columns:
            vpd = df['VPD']
        else:
            es_max = 0.6108 * np.exp(17.27 * df['TMAX'] / (df['TMAX'] + 237.3))
            es_min = 0.6108 * np.exp(17.27 * df['TMIN'] / (df['TMIN'] + 237.3))
            es = (es_max + es_min) / 2
            vap = df.get('VAP', es * 0.5)
            vpd = es - vap
        
        # Rayonnement net Rn ≈ 0.77 * Rs (où Rs = IRRAD en MJ/m²/j)
        # IRRAD est déjà en kJ/m², converti plus tôt
        Rn = 0.77 * df['IRRAD'] / 1000  # Approximation simple (MJ/m²/j)
        
        # Termes PM
        wind_ms = df['WIND'] / 3.6
        numer = 0.408 * Delta * Rn + gamma * (900 / (t_mean + 273)) * wind_ms * vpd
        denom = Delta + gamma * (1 + 0.34 * wind_ms)
        df['ET0'] = np.where(denom != 0, numer / denom, np.nan)

    return df

def load_csv(csv_path: str, df_master = None, depth = None) -> pd.DataFrame:
    df = pd.read_csv(csv_path, low_memory=False)

    # Standardisation du nom de colonne date
    if 'date' in df.columns:
        df = df.rename(columns={'date': DATE_COL})
    if 'Unnamed: 0' in df.columns and DATE_COL not in df.columns:
        df = df.rename(columns={'Unnamed: 0': DATE_COL})

    if DATE_COL not in df.columns:
        raise ValueError(f"[{csv_path}] Colonne date ({DATE_COL}) manquante")

    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

    for c in FEATURE_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")

    # --- RESAMPLE d'abord (horaire → journalier) ---
    df = df.set_index(DATE_COL)
    if len(df) > 0 and (df.index.value_counts().mean() > 1.5):
        df = resample_timeseries(df, freq='D', method='mean')
    df = df.reset_index()

    # --- MERGE météo ensuite ---
    station_dir = os.path.dirname(csv_path)
    meteo_path = os.path.join(station_dir, 'meteo_daily.csv')
    if os.path.exists(meteo_path):
        df_meteo = pd.read_csv(meteo_path)
        df_meteo[DATE_COL] = pd.to_datetime(df_meteo[DATE_COL])
        for c in DENSE_FEATURE_COLS:
            if c in df_meteo.columns:
                df_meteo[c] = pd.to_numeric(df_meteo[c], errors="coerce")
        df = pd.merge(df, df_meteo, on=DATE_COL, how='inner', suffixes=('', '_meteo'))
    
    if TARGET_COL in df.columns:
        df = interpolate_timeseries(df, col=TARGET_COL, n=1)

    df = update_soil_property(df, df_master, SOIL_PROPERTIES_COLS, depth)
    if df is None:
        return None
    
    df = engineer_features(df)
    
    # --- Filtrage colonnes ---
    cols_to_keep = [DATE_COL] + SOIL_PROPERTIES_COLS + DENSE_FEATURE_COLS 
    if TARGET_COL not in cols_to_keep:
        cols_to_keep.append(TARGET_COL)
    for c in SPARSE_FEATURE_COLS :
        if c in df.columns:
            cols_to_keep.append(c)

    missing = [c for c in cols_to_keep if c not in df.columns]
    if missing:
        raise ValueError(f"[{csv_path}] Colonnes manquantes dans le final: {missing}")

    df = df[cols_to_keep].copy()
    df = df.sort_values(DATE_COL).reset_index(drop=True)
    return df

In [5]:

def get_file_paths(files_path: str) -> List[str]:
    """Parcourt depth_X/station_Y/ et retourne tous les *soil_moisture*.csv"""
    file_paths = []
    for station_dir in os.listdir(files_path):
        station_path = os.path.join(files_path, station_dir)
        if not os.path.isdir(station_path):
            continue
        for fname in os.listdir(station_path):
            if fname.endswith('.csv') and 'soil_moisture' in fname:
                file_paths.append(os.path.join(station_path, fname))
    return file_paths

def split_spatial_files(file_paths: List[str], df_master: pd.DataFrame, depth: float) -> Tuple[List[pd.DataFrame], List[pd.DataFrame], List[pd.DataFrame]]:

    # Grouper les chemins par station
    station_map = {}
    for path in file_paths:
        sid = os.path.basename(os.path.dirname(path))  # "station_1"
        station_map.setdefault(sid, []).append(path)

    # Shuffle des stations (pas des fichiers)
    station_ids = list(station_map.keys())
    np.random.shuffle(station_ids)

    n = len(station_ids)
    n_train = max(1, int(n * 0.7))
    n_val   = max(1, int(n * 0.15))

    train_stations = station_ids[:n_train]
    val_stations   = station_ids[n_train:n_train + n_val]
    test_stations  = station_ids[n_train + n_val:]

    def load_station_group(station_ids_subset):
            result = []
            for sid in station_ids_subset:
                for p in station_map[sid]:
                    df = load_csv(p, df_master, depth)  
                    if df is None:
                        continue
                    if len(df) >= max(LOOKBACK) + max(HORIZONS) + 1:
                        result.append(df)
            return result

    train_list = load_station_group(train_stations)
    val_list   = load_station_group(val_stations)
    test_list  = load_station_group(test_stations)


    print(f"Stations: train={len(train_stations)}, val={len(val_stations)}, test={len(test_stations)}")
    print(f"Fichiers chargés: train={len(train_list)}, val={len(val_list)}, test={len(test_list)}")
    return train_list, val_list, test_list

def count_valid_windows(df: pd.DataFrame, lookback: int, horizon: int) -> int:
    # Vérifie où il n'y a pas de NaN dans DENSE_FEATURE_COLS et TARGET_COL
    mask_dense = df[DENSE_FEATURE_COLS].notna().all(axis=1)
    mask_soil = df[SOIL_PROPERTIES_COLS].notna().all(axis=1)
    mask_target = df[[TARGET_COL]].notna().all(axis=1)
    mask = mask_dense & mask_soil & mask_target

    # Compte le nombre de fenêtres valides
    valid_indices = np.where(mask)[0]
    count = max(0, len(valid_indices) - lookback - horizon + 1)
    return count

def count_valid_sparse_windows(df, lookback, horizon, sparse_cols):
    n = len(df)
    starts = np.arange(lookback, n - horizon + 1)
    # Vérifier si les colonnes sparses sont valides à l'index X (start - lookback)
    return df[sparse_cols].iloc[starts - lookback].notna().all(axis=1).sum()

def cut_timeseries(df, min_length=0):
    """
    Découpe une série temporelle (DataFrame) en séquences (DataFrames) sans NaN pour LSTM.
    min_length permet d'ignorer les séquences qui sont trop courtes.
    """
    sequences = []
    mask_tot = None
    # Identifier les valeurs valides
    for col in df.columns:
        if col in DENSE_FEATURE_COLS + SOIL_PROPERTIES_COLS + [TARGET_COL]:
            mask = df[col].notna()
            mask_tot = mask if mask_tot is None else mask_tot & mask
    
    # Créer un identifiant de groupe qui s'incrémente à chaque présence de NaN
    groups = (~mask_tot).cumsum()
    
    # Grouper les données valides par l'identifiant et ajouter chaque sous-dataframe
    for _, group in df[mask_tot].groupby(groups):
        if not group.empty and len(group) >= min_length:
            sequences.append(group)
    
    return sequences

def cut_and_filter_dfs(df_list, lookback, horizon, max_windows):
    all_segments = []
    for df in df_list:
        segments = cut_timeseries(df, min_length=lookback + horizon + 1)
        all_segments.extend(segments)
    if not all_segments:
        return []

    selected = []
    remaining = max_windows

    # Mélanger les segments pour éviter de toujours prendre les mêmes blocs
    rng = np.random.default_rng(SEED)
    rng.shuffle(all_segments)

    # ── Mode sparse : sélectionner des fenêtres individuelles de façon aléatoire ──
    if SPARSE_FEATURE_COLS and lookback == 1:
        for seg in all_segments:
            if remaining <= 0:
                break
            starts = np.arange(lookback, len(seg) - horizon + 1)
            if len(starts) == 0:
                continue
            valid = seg[SPARSE_FEATURE_COLS].iloc[starts - lookback].notna().all(axis=1)
            valid_starts = starts[valid]
            if len(valid_starts) == 0:
                continue
            rng.shuffle(valid_starts)
            for s in valid_starts:
                if remaining <= 0:
                    break
                selected.append(seg.iloc[s - lookback : s + horizon])
                remaining -= 1
        return selected

    # ── Mode normal : sous-échantillonnage aléatoire des fenêtres valides ──
    candidate_windows = []
    for seg in all_segments:
        n_windows = len(seg) - lookback - horizon + 1
        if n_windows <= 0:
            continue
        for start in range(n_windows):
            candidate_windows.append(seg.iloc[start : start + lookback + horizon])

    if not candidate_windows:
        return []

    rng.shuffle(candidate_windows)
    return candidate_windows[:max_windows]

## Dataset local -> vecteurs d'entrainement (remplace l'extraction Earth Engine)

Cette cellule reconstruit, a partir des CSV locaux (au lieu de GEE), les memes
6 fichiers .npy que la cellule "Parameter tuning" chargeait auparavant :
(sentinel1 / sentinel2 / HLSL30) x (surface / rootzone).

Hypotheses faites ici, A VERIFIER / ADAPTER selon ton vrai layout local :
  - ML1 <-> features Sentinel-1 (FULL_SPARSE_S1), ML2 <-> Sentinel-2 (FULL_SPARSE_S2)
  - ML3 (HLSL30 a l'origine) n'a pas d'equivalent local -> on reprend S1+S2 (FULL_SPARSE)
    en attendant que tu branches une vraie source HLS si tu en as une.
  - "surface"/"rootzone" <-> DEPTHS[0] / DEPTHS[1]. Si tu n'as qu'une seule profondeur
    disponible localement, les deux pointeront sur la meme donnee (DEPTH_ROOTZONE
    retombe sur DEPTHS[0]) -- dans ce cas la distinction sly/rly n'a plus de sens et
    tu peux simplifier le pipeline plus bas (garder un seul bloc au lieu de 6).
  - Les CSV par station sont cherches directement sous base_path (pas de sous-dossier
    par profondeur) ; adapte `resolve_depth_folder` si tes CSV sont ranges dans des
    sous-dossiers du type depth_0.1/, depth_0.6/, etc.

In [6]:
df_master = pd.read_csv(MASTER_CSV_PATH)

ML_VARIANT_SPARSE_COLS = {
    "sentinel1": FULL_SPARSE_S1,   # -> ML1
    "sentinel2": FULL_SPARSE_S2,   # -> ML2
    "HLSL30":    FULL_SPARSE_HLS30,      # -> ML3 (pas de source HLS locale : repli sur S1+S2)
}

DEPTH_SURFACE = DEPTHS[0]
DEPTH_ROOTZONE = DEPTHS[1] if len(DEPTHS) > 1 else DEPTHS[0]

FILE_MAP = {
    ("sentinel1", "surface"):  ("sentinel1_surface_X_valid.npy", "sentinel1_surface_y_valid.npy"),
    ("sentinel1", "rootzone"): ("sentinel1_rootzone_X_valid.npy", "sentinel1_rootzone_y_valid.npy"),
    ("sentinel2", "surface"):  ("sentinel2_surface_X_valid.npy", "sentinel2_surface_y_valid.npy"),
    ("sentinel2", "rootzone"): ("sentinel2_rootzone_X_valid.npy", "sentinel2_rootzone_y_valid.npy"),
    ("HLSL30", "surface"):     ("HLSL30_surface_X_valid.npy", "HLSL30_surface_y_valid.npy"),
    ("HLSL30", "rootzone"):    ("HLSL30_rootzone_X_valid.npy", "HLSL30_rootzone_y_valid.npy"),
}


def resolve_depth_folder(depth):
    """Renvoie le dossier contenant les CSV station pour une profondeur donnee.
    Par defaut on suppose que tous les CSV de base_path sont deja a la (seule)
    profondeur consideree. Adapte cette fonction si tu as des sous-dossiers
    depth_0.1/, depth_0.6/, ... sous base_path."""

    return base_path + "_" + str(depth)


def load_local_dataset(depth, sparse_cols):
    """Charge tous les CSV station pour une profondeur donnee et construit
    un DataFrame combine (une ligne = une station/un jour)."""
    global DENSE_FEATURE_COLS, SOIL_PROPERTIES_COLS, SPARSE_FEATURE_COLS, FEATURE_COLS
    DENSE_FEATURE_COLS = FULL_DENSE
    SOIL_PROPERTIES_COLS = FULL_SOIL
    SPARSE_FEATURE_COLS = sparse_cols
    FEATURE_COLS = DENSE_FEATURE_COLS + SOIL_PROPERTIES_COLS + SPARSE_FEATURE_COLS

    depth_folder = resolve_depth_folder(depth)
    if not os.path.isdir(depth_folder):
        raise FileNotFoundError(
            f"Dossier introuvable: {depth_folder} -- adapte resolve_depth_folder() "
            f"a ton vrai layout local."
        )

    file_paths = get_file_paths(depth_folder)
    if not file_paths:
        raise FileNotFoundError(f"Aucun CSV *soil_moisture* trouve dans {depth_folder}")

    frames = []
    for p in file_paths:
        station_id = os.path.basename(os.path.dirname(p))
        if station_id not in SITE_META:
            raw_head = pd.read_csv(p, nrows=1)
            SITE_META[station_id] = (raw_head['Latitude'].iloc[0], raw_head['Longitude'].iloc[0])
        df = load_csv(p, df_master, depth)
        if df is None or df.empty:
            continue
        df["site_id"] = station_id
        frames.append(df)

    if not frames:
        raise ValueError(f"Aucune donnee valide chargee pour depth={depth}")

    return pd.concat(frames, ignore_index=True)


def build_X_y(df):
    """Transforme le DataFrame combine en tableaux X (4 colonnes info + features) et y.
    Les 4 colonnes info [site_id_numerique, annee, jour_de_l'annee, mois] remplacent les
    anciennes colonnes info issues de GEE -- elles sont retirees plus bas via
    removed_indices_MLx = [0, 1, 2, 3] avant l'entrainement."""
    df = df.dropna(subset=FEATURE_COLS + [TARGET_COL]).copy()

    site_codes = df["site_id"].map(SITE_CODES).to_numpy()
    year = df[DATE_COL].dt.year.to_numpy()
    doy = df[DATE_COL].dt.dayofyear.to_numpy()
    month = df[DATE_COL].dt.month.to_numpy()

    info = np.column_stack([site_codes, year, doy, month]).astype(np.float64)
    features = df[FEATURE_COLS].to_numpy(dtype=np.float64)
    X = np.hstack([info, features])
    y = df[TARGET_COL].to_numpy(dtype=np.float64).reshape(-1, 1)
    return X, y


# ----------------------------------------------------------------------------
# Codes de site STABLES (identiques pour les 6 combinaisons S1/S2/HLS30 x sly/rly).
# Le pd.factorize par combinaison (cf. build_X_y) donnait des codes DIFFERENTS
# selon l'ordre d'apparition des stations dans chaque variante -> la table de
# metriques par site (cellule "Calculate evaluation metrics") melangeait alors des
# stations differentes. On enumere donc toutes les stations une seule fois, triees,
# et on reutilise ces codes partout (build_X_y + merged_df).
SITE_CODES = {}
SITE_META = {}
for d in sorted(set([DEPTH_SURFACE, DEPTH_ROOTZONE])):
    for p in get_file_paths(resolve_depth_folder(d)):
        sid = os.path.basename(os.path.dirname(p))
        SITE_CODES.setdefault(sid, len(SITE_CODES))
        if sid not in SITE_META:
            raw_head = pd.read_csv(p, nrows=1)
            SITE_META[sid] = (raw_head['Latitude'].iloc[0], raw_head['Longitude'].iloc[0])

for variant, sparse_cols in ML_VARIANT_SPARSE_COLS.items():
    for layer, depth in [("surface", DEPTH_SURFACE), ("rootzone", DEPTH_ROOTZONE)]:
        x_name, y_name = FILE_MAP[(variant, layer)]
        if os.path.exists(x_name) and os.path.exists(y_name):
            print(f"[SKIP] {variant}/{layer}: {x_name} deja present, generation ignoree")
            continue
        df_combined = load_local_dataset(depth, sparse_cols)
        X, y = build_X_y(df_combined)
        np.save(x_name, X)
        np.save(y_name, y)
        print(f"{variant}/{layer}: X{X.shape} y{y.shape} -> {x_name}, {y_name}")


[SKIP] sentinel1/surface: sentinel1_surface_X_valid.npy deja present, generation ignoree
sentinel1/rootzone: X(37025, 32) y(37025, 1) -> sentinel1_rootzone_X_valid.npy, sentinel1_rootzone_y_valid.npy
[SKIP] sentinel2/surface: sentinel2_surface_X_valid.npy deja present, generation ignoree
sentinel2/rootzone: X(7770, 42) y(7770, 1) -> sentinel2_rootzone_X_valid.npy, sentinel2_rootzone_y_valid.npy
[SKIP] HLSL30/surface: HLSL30_surface_X_valid.npy deja present, generation ignoree
HLSL30/rootzone: X(1921, 37) y(1921, 1) -> HLSL30_rootzone_X_valid.npy, HLSL30_rootzone_y_valid.npy


## ML parameter tune

After the pre-process, we got the X and y for building ML models. In this section, we are going to -


1.   Parameter tune
2. Cross-validation
3.   Train final ML models and save it for use.


when tuning parameter, we use a self-defined 5-fold cross validtion, and these 5 folds are divided spatially based on site ID. This can ensure a better spatial transferability for ML models by using the parameters.

In total, we have six ML models: (Sentinel1, Sentinel2, HLSL30) X (surface, rootzone)

In [7]:

#@title Define parameter_tune_LGB function
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import numpy as np
import random
import json
import optuna



def parameter_tune_LGB(X, y, input_info_valid):
    id_arr = np.unique(input_info_valid[:, 0])
    print(id_arr.shape)
    random.shuffle(id_arr)
    id_arr_group = np.array_split(id_arr, 5)

    custom_cv = []
    for i in range(5):
        mask = np.isin(input_info_valid[:, 0], id_arr_group[i])  # test mask
        custom_cv.append((np.where(~mask)[0], np.where(mask)[0]))

    # sample weights
    y_true = y.copy().flatten()
    sample_weights = np.ones_like(y_true)
    site_ids = input_info_valid[:, 0]
    for sid in np.unique(site_ids):
        q_low = np.quantile(y_true[site_ids == sid], 0.20)
        q_high = np.quantile(y_true[site_ids == sid], 0.80)
        sample_weights[(site_ids == sid) & (y_true < q_low)] = 2
        sample_weights[(site_ids == sid) & (y_true > q_high)] = 2


    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # ------------- Optuna objective -------------
    def objective(trial):
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'verbosity': -1,
            'n_jobs': -1,

            'num_leaves': trial.suggest_int('num_leaves', 15, 127),
            'max_depth': trial.suggest_int('max_depth', -1, 20),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        }

        model = lgb.LGBMRegressor(**params)

        cv_rmse = []

        for train_idx, valid_idx in custom_cv:
            X_train, X_valid = X[train_idx], X[valid_idx]
            y_train, y_valid = y[train_idx], y[valid_idx]
            w_train = sample_weights[train_idx]

            model.fit(
                X_train, y_train,
                sample_weight=w_train,
            )

            preds = model.predict(X_valid)
            rmse = np.sqrt(mean_squared_error(y_valid, preds))
            cv_rmse.append(rmse)

        return np.mean(cv_rmse)

    # ------------- Run Optuna -------------
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=50)

    print(f"Best parameters: {study.best_params}")
    print(f"Best CV RMSE: {study.best_value:.4f}")

    return study.best_params


In [8]:
#@title Parameter tuning
# Avec le pipeline local, les 4 premieres colonnes de X sont toujours
# [site_id, annee, jour_de_l'annee, mois] (cf. build_X_y ci-dessus) -> on les retire
# systematiquement avant l'entrainement, plus besoin des indices specifiques a GEE.
removed_indices_ML1 = [0, 1, 2, 3]
removed_indices_ML2 = [0, 1, 2, 3]
removed_indices_ML3 = [0, 1, 2, 3]


ML1_sly_input_valid = np.load('sentinel1_surface_X_valid.npy')
ML1_sly_y_valid = np.load('sentinel1_surface_y_valid.npy')
ML2_sly_input_valid = np.load('sentinel2_surface_X_valid.npy')
ML2_sly_y_valid = np.load('sentinel2_surface_y_valid.npy')
ML3_sly_input_valid = np.load('HLSL30_surface_X_valid.npy')
ML3_sly_y_valid = np.load('HLSL30_surface_y_valid.npy')
ML1_rly_input_valid = np.load('sentinel1_rootzone_X_valid.npy')
ML1_rly_y_valid = np.load('sentinel1_rootzone_y_valid.npy')
ML2_rly_input_valid = np.load('sentinel2_rootzone_X_valid.npy')
ML2_rly_y_valid = np.load('sentinel2_rootzone_y_valid.npy')
ML3_rly_input_valid = np.load('HLSL30_rootzone_X_valid.npy')
ML3_rly_y_valid = np.load('HLSL30_rootzone_y_valid.npy')

print(ML1_sly_input_valid.shape, ML1_sly_y_valid.shape)
print(ML2_sly_input_valid.shape, ML2_sly_y_valid.shape)
print(ML3_sly_input_valid.shape, ML3_sly_y_valid.shape)
print(ML1_rly_input_valid.shape, ML1_rly_y_valid.shape)
print(ML2_rly_input_valid.shape, ML2_rly_y_valid.shape)
print(ML3_rly_input_valid.shape, ML3_rly_y_valid.shape)

ML1_sly_input_info_valid = ML1_sly_input_valid[:, 0:4].copy()
ML2_sly_input_info_valid = ML2_sly_input_valid[:, 0:4].copy()
ML3_sly_input_info_valid = ML3_sly_input_valid[:, 0:4].copy()
ML1_rly_input_info_valid = ML1_rly_input_valid[:, 0:4].copy()
ML2_rly_input_info_valid = ML2_rly_input_valid[:, 0:4].copy()
ML3_rly_input_info_valid = ML3_rly_input_valid[:, 0:4].copy()

ML1_sly_input_valid = np.delete(ML1_sly_input_valid, removed_indices_ML1, axis=1)
ML2_sly_input_valid = np.delete(ML2_sly_input_valid, removed_indices_ML2, axis=1)  # remove B2, B3, B4,, 24, 25, 26
ML3_sly_input_valid = np.delete(ML3_sly_input_valid, removed_indices_ML3,
                                axis=1)  # remove B2, B3, B4, SZA, SAA, VZA, VAA, 24, 25, 26, 32, 33, 34, 35
ML1_rly_input_valid = np.delete(ML1_rly_input_valid, removed_indices_ML1, axis=1)
ML2_rly_input_valid = np.delete(ML2_rly_input_valid, removed_indices_ML2, axis=1)  # , 24, 25, 26
ML3_rly_input_valid = np.delete(ML3_rly_input_valid, removed_indices_ML3, axis=1)  # , 24, 25, 26, 32, 33, 34, 35

print('After delete some features: ')
print(ML1_sly_input_valid.shape, ML1_sly_y_valid.shape)
print(ML2_sly_input_valid.shape, ML2_sly_y_valid.shape)
print(ML3_sly_input_valid.shape, ML3_sly_y_valid.shape)
print(ML1_rly_input_valid.shape, ML1_rly_y_valid.shape)
print(ML2_rly_input_valid.shape, ML2_rly_y_valid.shape)
print(ML3_rly_input_valid.shape, ML3_rly_y_valid.shape)

ML1_sly_y_valid = ML1_sly_y_valid.ravel()
ML2_sly_y_valid = ML2_sly_y_valid.ravel()
ML3_sly_y_valid = ML3_sly_y_valid.ravel()
ML1_rly_y_valid = ML1_rly_y_valid.ravel()
ML2_rly_y_valid = ML2_rly_y_valid.ravel()
ML3_rly_y_valid = ML3_rly_y_valid.ravel()

print('After reshape: ')
print(ML1_sly_input_valid.shape, ML1_sly_y_valid.shape)
print(ML2_sly_input_valid.shape, ML2_sly_y_valid.shape)
print(ML3_sly_input_valid.shape, ML3_sly_y_valid.shape)
print(ML1_rly_input_valid.shape, ML1_rly_y_valid.shape)
print(ML2_rly_input_valid.shape, ML2_rly_y_valid.shape)
print(ML3_rly_input_valid.shape, ML3_rly_y_valid.shape)

# LGB ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Version "reprise" : on ne relance que les tunings dont le fichier
# *lgb_reg_best_params.json n'existe pas encore sur Google Drive
# (Local_Training). Les 3 deja termines (ML1_sly, ML1_rly, ML2_sly) sont
# charges depuis leur JSON -> on ne retune que ML2_rly, ML3_sly, ML3_rly.
TUNING_VARIANTS = [
    ("ML1_sly", ML1_sly_input_valid, ML1_sly_y_valid, ML1_sly_input_info_valid),
    ("ML1_rly", ML1_rly_input_valid, ML1_rly_y_valid, ML1_rly_input_info_valid),
    ("ML2_sly", ML2_sly_input_valid, ML2_sly_y_valid, ML2_sly_input_info_valid),
    ("ML2_rly", ML2_rly_input_valid, ML2_rly_y_valid, ML2_rly_input_info_valid),
    ("ML3_sly", ML3_sly_input_valid, ML3_sly_y_valid, ML3_sly_input_info_valid),
    ("ML3_rly", ML3_rly_input_valid, ML3_rly_y_valid, ML3_rly_input_info_valid),
]

for name, X, y, info in TUNING_VARIANTS:
    params_path = f"{name}_lgb_reg_best_params.json"
    if os.path.exists(params_path):
        with open(params_path) as f:
            params = json.load(f)
        print(f"[SKIP] {name}: {params_path} deja present -> params charges")
    else:
        print(f"[TUNE] {name}: lancement du tuning Optuna (50 essais)...")
        params = parameter_tune_LGB(X, y, info)
        with open(params_path, "w") as f:
            json.dump(params, f)
        print(f"[SAVE] {name}: best params ecrits dans {params_path}")
    globals()[f"{name}_lgb_reg_best_params"] = params

(114652, 32) (114652, 1)
(27096, 42) (27096, 1)
(6952, 37) (6952, 1)
(37025, 32) (37025, 1)
(7770, 42) (7770, 1)
(1921, 37) (1921, 1)
After delete some features: 
(114652, 28) (114652, 1)
(27096, 38) (27096, 1)
(6952, 33) (6952, 1)
(37025, 28) (37025, 1)
(7770, 38) (7770, 1)
(1921, 33) (1921, 1)
After reshape: 
(114652, 28) (114652,)
(27096, 38) (27096,)
(6952, 33) (6952,)
(37025, 28) (37025,)
(7770, 38) (7770,)
(1921, 33) (1921,)


[I 2026-08-10 14:10:12,511] A new study created in memory with name: no-name-5437f303-c71a-4e99-b4e8-1d218b75475a


[SKIP] ML1_sly: ML1_sly_lgb_reg_best_params.json deja present -> params charges
[TUNE] ML1_rly: lancement du tuning Optuna (50 essais)...
(20,)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-08-10 14:10:57,307] Trial 0 finished with value

Best parameters: {'num_leaves': 27, 'max_depth': -1, 'learning_rate': 0.013335431181131696, 'n_estimators': 102, 'min_child_samples': 12, 'subsample': 0.900247242354628, 'colsample_bytree': 0.667326526769623, 'reg_alpha': 0.7991129452072895, 'reg_lambda': 0.9360974551251255}
Best CV RMSE: 0.1175
[SAVE] ML1_rly: best params ecrits dans ML1_rly_lgb_reg_best_params.json


...

Best parameters: {'num_leaves': 49, 'max_depth': 8, 'learning_rate': 0.04207458789850553, 'n_estimators': 169, 'min_child_samples': 46, 'subsample': 0.7695685071019309, 'colsample_bytree': 0.9074104424162498, 'reg_alpha': 0.7205501828802202, 'reg_lambda': 0.1822864602684095}
Best CV RMSE: 0.1358
[SAVE] ML2_rly: best params ecrits dans ML2_rly_lgb_reg_best_params.json


[I 2026-08-10 14:31:15,801] A new study created in memory with name: no-name-7253a9ff-567c-49b2-8698-68fa4e4afbc8


[SKIP] ML3_sly: ML3_sly_lgb_reg_best_params.json deja present -> params charges
[TUNE] ML3_rly: lancement du tuning Optuna (50 essais)...
(20,)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-08-10 14:31:18,057] Trial 0 finished with value

Best parameters: {'num_leaves': 23, 'max_depth': 18, 'learning_rate': 0.02918746841982575, 'n_estimators': 797, 'min_child_samples': 16, 'subsample': 0.6855378041429961, 'colsample_bytree': 0.9399608023310919, 'reg_alpha': 0.10029449161273846, 'reg_lambda': 0.3293004384788415}
Best CV RMSE: 0.1266
[SAVE] ML3_rly: best params ecrits dans ML3_rly_lgb_reg_best_params.json


## Cross validation of ML models

In [9]:
import os
import lightgbm as lgb
import numpy as np
from joblib import dump, load
from sklearn.preprocessing import StandardScaler
import random
import json
from permetrics import RegressionMetric
from mapie.metrics.regression import regression_coverage_score
from mapie.regression import SplitConformalRegressor, ConformalizedQuantileRegressor
RANDOM_STATE = 1
confidence_level = 0.9


# Fonction de metriques (corr, bias, RMSE, ubRMSE, KGE) qui manquait dans le
# pipeline (NameError sinon). Retourne un pd.Series au ordre FIXE :
# corr, bias, RMSE, ubRMSE, KGE (les .values() plus bas les deplient dans cet ordre).
# Jeux vides ou constants -> NaN au lieu de crasher.
def accuracy(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    res = {'corr': np.nan, 'bias': np.nan, 'RMSE': np.nan, 'ubRMSE': np.nan, 'KGE': np.nan}
    if len(y_true) < 2 or len(y_pred) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return res
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    res['corr'] = corr
    res['bias'] = float(np.mean(y_pred) - np.mean(y_true))
    res['RMSE'] = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    res['ubRMSE'] = float(np.sqrt(np.mean(
        ((y_pred - y_pred.mean()) - (y_true - y_true.mean())) ** 2)))
    alpha = np.std(y_pred) / np.std(y_true)
    beta = np.mean(y_pred) / np.mean(y_true) if np.mean(y_true) != 0 else np.nan
    res['KGE'] = 1.0 - np.sqrt((corr - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)
    return res


def cross_valid_and_pred_LGB(id_groups, ML_params, ML_input_valid, ML_input_info_valid, ML_y_valid, ML_input_all, ML_input_info_all):
    ML_y_valid = ML_y_valid.ravel()
    # sample weights
    y_true = ML_y_valid.copy().flatten()
    sample_weights = np.ones_like(y_true)
    site_ids = ML_input_info_valid[:, 0]
    for sid in np.unique(site_ids):
        q_low = np.quantile(y_true[site_ids == sid], 0.20)
        q_high = np.quantile(y_true[site_ids == sid], 0.80)
        sample_weights[(site_ids == sid) & (y_true < q_low)] = 2
        sample_weights[(site_ids == sid) & (y_true > q_high)] = 2

    ml_result_valid_arr = np.array([])
    ml_result_all_arr = np.array([])
    for i in range(len(id_groups)):
        # train
        testing_sites = id_groups[i]
        clib_sites = id_groups[(i+1)%8]
        testing_indice_valid = np.isin(ML_input_info_valid[:, 0], testing_sites)
        clib_indice_valid = np.isin(ML_input_info_valid[:, 0], clib_sites)
        training_indice_valid = ~(testing_indice_valid | clib_indice_valid)
        print(f'total: {len(testing_indice_valid)}; testing size: {np.count_nonzero(testing_indice_valid)}; '
              f'clibration size: {np.count_nonzero(clib_indice_valid)}; '
              f'training size: {np.count_nonzero(training_indice_valid)}')

        X_train_valid = ML_input_valid[training_indice_valid, :]
        y_train_valid = ML_y_valid[training_indice_valid]
        X_calib_valid = ML_input_valid[clib_indice_valid, :]
        y_calib_valid = ML_y_valid[clib_indice_valid]
        X_test_valid = ML_input_valid[testing_indice_valid, :]
        X_info_test_valid = ML_input_info_valid[testing_indice_valid, :]
        y_test_valid = ML_y_valid[testing_indice_valid]
        sample_weights_train = sample_weights[training_indice_valid]

        # scale transform
        scaler = StandardScaler()
        X_train_valid_scaled = scaler.fit_transform(X_train_valid)
        X_calib_valid_scaled = scaler.transform(X_calib_valid)
        X_test_valid_scaled = scaler.transform(X_test_valid)

        # ML1 - surface layer
        best_params = ML_params[0]
        final_model = lgb.LGBMRegressor(**best_params, verbosity =-1)
        final_model.fit(X_train_valid_scaled, y_train_valid, sample_weight=sample_weights_train)
        y_test_pred = final_model.predict(X_test_valid_scaled)
        y_train_pred = final_model.predict(X_train_valid_scaled)

        # testing accuracy
        test_scores = accuracy(y_test_valid.reshape(-1), y_test_pred)
        # print results
        scores_msg = ", ".join([f"{k}={v:.4f}" for (k, v) in test_scores.items()])
        print(f"Test results : \n{scores_msg}")
        print("\t".join([f"{v:.4f}" for v in test_scores.values()]), "\n\n")

        # training accuracy
        train_scores = accuracy(y_train_valid.reshape(-1), y_train_pred)
        # print results
        scores_msg = ", ".join([f"{k}={v:.4f}" for (k, v) in train_scores.items()])
        print(f"Train results : \n{scores_msg}")
        print("\t".join([f"{v:.4f}" for v in train_scores.values()]), "\n\n")

        # train the quantiles
        # Train LGBMRegressor models for _MapieQuantileRegressor
        list_estimators_cqr = []
        for alpha_ in [(1 - confidence_level) / 2, (1 + confidence_level) / 2, 0.5]:
            estimator_ = lgb.LGBMRegressor(**best_params,
                objective='quantile',
                alpha=alpha_, verbosity = -1
            )
            estimator_.fit(X_train_valid_scaled, y_train_valid)
            list_estimators_cqr.append(estimator_)

        # Conformalize uncertainties on conformalize set
        mapie_cqr = ConformalizedQuantileRegressor(
            list_estimators_cqr, confidence_level=0.9, prefit=True
        )
        mapie_cqr.conformalize(X_calib_valid_scaled, y_calib_valid)

        # Evaluate prediction and coverage level on testing set
        y_pred_cqr, y_pis_cqr = mapie_cqr.predict_interval(X_test_valid_scaled)
        coverage_cqr = regression_coverage_score(
            y_test_valid,
            y_pis_cqr
        )[0]
        print(f'coverage: {coverage_cqr}')
        print()


        #### ----------------------predict testing sites of valid
        y_hat = final_model.predict(X_test_valid_scaled)
        y_up_hat = y_pis_cqr[:, 1, 0]
        y_median_hat = y_pred_cqr
        y_low_hat = y_pis_cqr[:, 0, 0]

        ml_result_valid_arr_grp = np.column_stack((X_info_test_valid, y_hat, y_up_hat, y_median_hat, y_low_hat, y_test_valid))


        ###----------------predict testing sites of all
        testing_indice_all = np.isin(ML_input_info_all[:, 0], testing_sites)

        X_test_all = ML_input_all[testing_indice_all, :]
        X_info_test_all = ML_input_info_all[testing_indice_all, :]

        X_test_all_scaled = scaler.transform(X_test_all)

        y_hat = final_model.predict(X_test_all_scaled)
        y_pred, y_pis = mapie_cqr.predict_interval(X_test_all_scaled)
        y_up_hat = y_pis[:, 1, 0]
        y_median_hat = y_pred
        y_low_hat = y_pis[:, 0, 0]

        ml_result_all_arr_grp = np.column_stack((X_info_test_all, y_hat, y_up_hat, y_median_hat, y_low_hat))

        '''combine results'''
        if i == 0:
            ml_result_valid_arr = ml_result_valid_arr_grp.copy()
            ml_result_all_arr = ml_result_all_arr_grp.copy()
        else:
            ml_result_valid_arr = np.vstack((ml_result_valid_arr, ml_result_valid_arr_grp))
            ml_result_all_arr = np.vstack((ml_result_all_arr, ml_result_all_arr_grp))

    return ml_result_valid_arr, ml_result_all_arr

In [10]:
#@title Do cross validation
df_name = 'df_ML_metrics_removeSMAPB2B3B4Angle_ws'
# Avec le layout local, X = [site_id, annee, doy, mois] + features -> on retire
# toujours les 4 colonnes info (comme dans la cellule "Parameter tuning").
removed_indices_ML1 = [0, 1, 2, 3]
removed_indices_ML2 = [0, 1, 2, 3]
removed_indices_ML3 = [0, 1, 2, 3]


ML1_sly_input_valid = np.load('sentinel1_surface_X_valid.npy')
# NB: le pipeline local ne produit PAS de fichiers *_X_all.npy (chaque ligne a deja
# un y). On reutilise donc *_valid comme pseudo-"_all" ; seules les predictions _valid
# sont utilisees par la cellule de metriques.

ML1_sly_input_all = ML1_sly_input_valid  # option B: pas de dataset "all" en local
ML1_sly_y_valid = np.load('sentinel1_surface_y_valid.npy')
ML2_sly_input_valid = np.load('sentinel2_surface_X_valid.npy')
ML2_sly_input_all = ML2_sly_input_valid
ML2_sly_y_valid = np.load('sentinel2_surface_y_valid.npy')
ML3_sly_input_valid = np.load('HLSL30_surface_X_valid.npy')
ML3_sly_input_all = ML3_sly_input_valid
ML3_sly_y_valid = np.load('HLSL30_surface_y_valid.npy')
ML1_rly_input_valid = np.load('sentinel1_rootzone_X_valid.npy')
ML1_rly_input_all = ML1_rly_input_valid
ML1_rly_y_valid = np.load('sentinel1_rootzone_y_valid.npy')
ML2_rly_input_valid = np.load('sentinel2_rootzone_X_valid.npy')
ML2_rly_input_all = ML2_rly_input_valid
ML2_rly_y_valid = np.load('sentinel2_rootzone_y_valid.npy')
ML3_rly_input_valid = np.load('HLSL30_rootzone_X_valid.npy')
ML3_rly_input_all = ML3_rly_input_valid
ML3_rly_y_valid = np.load('HLSL30_rootzone_y_valid.npy')

print(ML1_sly_input_valid.shape, ML1_sly_y_valid.shape)
print(ML2_sly_input_valid.shape, ML2_sly_y_valid.shape)
print(ML3_sly_input_valid.shape, ML3_sly_y_valid.shape)
print(ML1_rly_input_valid.shape, ML1_rly_y_valid.shape)
print(ML2_rly_input_valid.shape, ML2_rly_y_valid.shape)
print(ML3_rly_input_valid.shape, ML3_rly_y_valid.shape)

ML1_sly_input_info_valid = ML1_sly_input_valid[:, 0:4].copy()
ML2_sly_input_info_valid = ML2_sly_input_valid[:, 0:4].copy()
ML3_sly_input_info_valid = ML3_sly_input_valid[:, 0:4].copy()
ML1_rly_input_info_valid = ML1_rly_input_valid[:, 0:4].copy()
ML2_rly_input_info_valid = ML2_rly_input_valid[:, 0:4].copy()
ML3_rly_input_info_valid = ML3_rly_input_valid[:, 0:4].copy()
ML1_sly_input_info_all = ML1_sly_input_all[:, 0:4].copy()
ML2_sly_input_info_all = ML2_sly_input_all[:, 0:4].copy()
ML3_sly_input_info_all = ML3_sly_input_all[:, 0:4].copy()
ML1_rly_input_info_all = ML1_rly_input_all[:, 0:4].copy()
ML2_rly_input_info_all = ML2_rly_input_all[:, 0:4].copy()
ML3_rly_input_info_all = ML3_rly_input_all[:, 0:4].copy()

ML1_sly_input_valid = np.delete(ML1_sly_input_valid, removed_indices_ML1, axis=1)  # , 15
ML2_sly_input_valid = np.delete(ML2_sly_input_valid, removed_indices_ML2, axis=1)
ML3_sly_input_valid = np.delete(ML3_sly_input_valid, removed_indices_ML3, axis=1)
ML1_rly_input_valid = np.delete(ML1_rly_input_valid, removed_indices_ML1, axis=1)
ML2_rly_input_valid = np.delete(ML2_rly_input_valid, removed_indices_ML2, axis=1)
ML3_rly_input_valid = np.delete(ML3_rly_input_valid, removed_indices_ML3, axis=1)
ML1_sly_input_all = np.delete(ML1_sly_input_all, removed_indices_ML1, axis=1)
ML2_sly_input_all = np.delete(ML2_sly_input_all, removed_indices_ML2, axis=1)
ML3_sly_input_all = np.delete(ML3_sly_input_all, removed_indices_ML3, axis=1)
ML1_rly_input_all = np.delete(ML1_rly_input_all, removed_indices_ML1, axis=1)
ML2_rly_input_all = np.delete(ML2_rly_input_all, removed_indices_ML2, axis=1)
ML3_rly_input_all = np.delete(ML3_rly_input_all, removed_indices_ML3, axis=1)

"""------------------------------LGB---------------------------------------------------"""

with open('ML1_sly_lgb_reg_best_params.json', 'r') as f:
    ML1_sly_lgb_reg_best_params = json.load(f)
with open('ML1_rly_lgb_reg_best_params.json', 'r') as f:
    ML1_rly_lgb_reg_best_params = json.load(f)
with open('ML2_sly_lgb_reg_best_params.json', 'r') as f:
    ML2_sly_lgb_reg_best_params = json.load(f)
with open('ML2_rly_lgb_reg_best_params.json', 'r') as f:
    ML2_rly_lgb_reg_best_params = json.load(f)
with open('ML3_sly_lgb_reg_best_params.json', 'r') as f:
    ML3_sly_lgb_reg_best_params = json.load(f)
with open('ML3_rly_lgb_reg_best_params.json', 'r') as f:
    ML3_rly_lgb_reg_best_params = json.load(f)

ML_params_dict = {
    'ML1_sly': [ML1_sly_lgb_reg_best_params,
                ],
    'ML1_rly': [ML1_rly_lgb_reg_best_params,
                ],
    'ML2_sly': [ML2_sly_lgb_reg_best_params,
                ],
    'ML2_rly': [ML2_rly_lgb_reg_best_params,
                ],
    'ML3_sly': [ML3_sly_lgb_reg_best_params,
                ],
    'ML3_rly': [ML3_rly_lgb_reg_best_params,
                ]
}

id_arr = np.unique(ML1_sly_input_info_valid[:, 0])
print(id_arr.shape)
random.shuffle(id_arr)
id_arr_group = np.array_split(id_arr, 8)

ML1_sly_y_pred_valid_arr, ML1_sly_y_pred_all_arr = cross_valid_and_pred_LGB(id_arr_group,
                                                                            ML_params_dict['ML1_sly'],
                                                                            ML1_sly_input_valid,
                                                                            ML1_sly_input_info_valid,
                                                                            ML1_sly_y_valid,
                                                                            ML1_sly_input_all,
                                                                            ML1_sly_input_info_all)

id_arr = np.unique(ML1_rly_input_info_valid[:, 0])
print(id_arr.shape)
random.shuffle(id_arr)
id_arr_group = np.array_split(id_arr, 8)
ML1_rly_y_pred_valid_arr, ML1_rly_y_pred_all_arr = cross_valid_and_pred_LGB(id_arr_group,
                                                                            ML_params_dict['ML1_rly'],
                                                                            ML1_rly_input_valid,
                                                                            ML1_rly_input_info_valid,
                                                                            ML1_rly_y_valid,
                                                                            ML1_rly_input_all,
                                                                            ML1_rly_input_info_all)

id_arr = np.unique(ML2_sly_input_info_valid[:, 0])
print(id_arr.shape)
random.shuffle(id_arr)
id_arr_group = np.array_split(id_arr, 8)
ML2_sly_y_pred_valid_arr, ML2_sly_y_pred_all_arr = cross_valid_and_pred_LGB(id_arr_group,
                                                                            ML_params_dict['ML2_sly'],
                                                                            ML2_sly_input_valid,
                                                                            ML2_sly_input_info_valid,
                                                                            ML2_sly_y_valid,
                                                                            ML2_sly_input_all,
                                                                            ML2_sly_input_info_all)

id_arr = np.unique(ML2_rly_input_info_valid[:, 0])
print(id_arr.shape)
random.shuffle(id_arr)
id_arr_group = np.array_split(id_arr, 8)
ML2_rly_y_pred_valid_arr, ML2_rly_y_pred_all_arr = cross_valid_and_pred_LGB(id_arr_group,
                                                                            ML_params_dict['ML2_rly'],
                                                                            ML2_rly_input_valid,
                                                                            ML2_rly_input_info_valid,
                                                                            ML2_rly_y_valid,
                                                                            ML2_rly_input_all,
                                                                            ML2_rly_input_info_all)

id_arr = np.unique(ML3_sly_input_info_valid[:, 0])
print(id_arr.shape)
random.shuffle(id_arr)
id_arr_group = np.array_split(id_arr, 8)
ML3_sly_y_pred_valid_arr, ML3_sly_y_pred_all_arr = cross_valid_and_pred_LGB(id_arr_group,
                                                                            ML_params_dict['ML3_sly'],
                                                                            ML3_sly_input_valid,
                                                                            ML3_sly_input_info_valid,
                                                                            ML3_sly_y_valid,
                                                                            ML3_sly_input_all,
                                                                            ML3_sly_input_info_all)

id_arr = np.unique(ML3_rly_input_info_valid[:, 0])
print(id_arr.shape)
random.shuffle(id_arr)
id_arr_group = np.array_split(id_arr, 8)
ML3_rly_y_pred_valid_arr, ML3_rly_y_pred_all_arr = cross_valid_and_pred_LGB(id_arr_group,
                                                                            ML_params_dict['ML3_rly'],
                                                                            ML3_rly_input_valid,
                                                                            ML3_rly_input_info_valid,
                                                                            ML3_rly_y_valid,
                                                                            ML3_rly_input_all,
                                                                            ML3_rly_input_info_all)

np.save('ML1_sly_y_pred_valid.npy', ML1_sly_y_pred_valid_arr)
np.save('ML1_rly_y_pred_valid.npy', ML1_rly_y_pred_valid_arr)
np.save('ML1_sly_y_pred_all.npy', ML1_sly_y_pred_all_arr)
np.save('ML1_rly_y_pred_all.npy', ML1_rly_y_pred_all_arr)
np.save('ML2_sly_y_pred_valid.npy', ML2_sly_y_pred_valid_arr)
np.save('ML2_rly_y_pred_valid.npy', ML2_rly_y_pred_valid_arr)
np.save('ML2_sly_y_pred_all.npy', ML2_sly_y_pred_all_arr)
np.save('ML2_rly_y_pred_all.npy', ML2_rly_y_pred_all_arr)
np.save('ML3_sly_y_pred_valid.npy', ML3_sly_y_pred_valid_arr)
np.save('ML3_rly_y_pred_valid.npy', ML3_rly_y_pred_valid_arr)
np.save('ML3_sly_y_pred_all.npy', ML3_sly_y_pred_all_arr)
np.save('ML3_rly_y_pred_all.npy', ML3_rly_y_pred_all_arr)

(114652, 32) (114652, 1)
(27096, 42) (27096, 1)
(6952, 37) (6952, 1)
(37025, 32) (37025, 1)
(7770, 42) (7770, 1)
(1921, 37) (1921, 1)
(61,)
total: 114652; testing size: 14958; clibration size: 13354; training size: 86340


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Test results : 
corr=0.4825, bias=0.0390, RMSE=0.1207, ubRMSE=0.1142, KGE=0.2750
0.4825	0.0390	0.1207	0.1142	0.2750 


Train results : 
corr=0.8601, bias=-0.0005, RMSE=0.0539, ubRMSE=0.0539, KGE=0.7107
0.8601	-0.0005	0.0539	0.0539	0.7107 




/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

coverage: 0.4173686321700762



...

coverage: 0.9739583333333334

total: 1921; testing size: 143; clibration size: 205; training size: 1573


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Test results : 
corr=0.3354, bias=-0.0933, RMSE=0.1239, ubRMSE=0.0815, KGE=0.1192
0.3354	-0.0933	0.1239	0.0815	0.1192 


Train results : 
corr=0.9900, bias=0.0004, RMSE=0.0157, ubRMSE=0.0156, KGE=0.9813
0.9900	0.0004	0.0157	0.0156	0.9813 




/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

coverage: 0.5034965034965035



In [11]:
#@title Calculate evaluation metrics

# merged_df etait charge depuis un CSV GEE dans le notebook d'origine ; en local on le
# reconstruit a partir des stations chargees dans la cellule "Dataset local".
merged_df = pd.DataFrame(
    [(sid, SITE_CODES[sid], lat, lon) for sid, (lat, lon) in SITE_META.items()],
    columns=['site_id', 'ID', 'Latitude', 'Longitude'],
).sort_values('ID').reset_index(drop=True)

refer_id_arr = merged_df['ID'].to_numpy()
refer_lat_arr = merged_df['Latitude'].to_numpy()
refer_lon_arr = merged_df['Longitude'].to_numpy()

idds = np.zeros(len(refer_id_arr))
ML1_sly_RMSEs = np.zeros(len(refer_id_arr))
ML1_rly_RMSEs = np.zeros(len(refer_id_arr))
ML2_sly_RMSEs = np.zeros(len(refer_id_arr))
ML2_rly_RMSEs = np.zeros(len(refer_id_arr))
ML3_sly_RMSEs = np.zeros(len(refer_id_arr))
ML3_rly_RMSEs = np.zeros(len(refer_id_arr))

ML1_sly_RMSEs[:] = np.nan
ML1_rly_RMSEs[:] = np.nan
ML2_sly_RMSEs[:] = np.nan
ML2_rly_RMSEs[:] = np.nan
ML3_sly_RMSEs[:] = np.nan
ML3_rly_RMSEs[:] = np.nan

ML1_sly_ubRMSEs = np.zeros(len(refer_id_arr))
ML1_rly_ubRMSEs = np.zeros(len(refer_id_arr))
ML2_sly_ubRMSEs = np.zeros(len(refer_id_arr))
ML2_rly_ubRMSEs = np.zeros(len(refer_id_arr))
ML3_sly_ubRMSEs = np.zeros(len(refer_id_arr))
ML3_rly_ubRMSEs = np.zeros(len(refer_id_arr))

ML1_sly_ubRMSEs[:] = np.nan
ML1_rly_ubRMSEs[:] = np.nan
ML2_sly_ubRMSEs[:] = np.nan
ML2_rly_ubRMSEs[:] = np.nan
ML3_sly_ubRMSEs[:] = np.nan
ML3_rly_ubRMSEs[:] = np.nan

ML1_sly_corrs = np.zeros(len(refer_id_arr))
ML1_rly_corrs = np.zeros(len(refer_id_arr))
ML2_sly_corrs = np.zeros(len(refer_id_arr))
ML2_rly_corrs = np.zeros(len(refer_id_arr))
ML3_sly_corrs = np.zeros(len(refer_id_arr))
ML3_rly_corrs = np.zeros(len(refer_id_arr))

ML1_sly_corrs[:] = np.nan
ML1_rly_corrs[:] = np.nan
ML2_sly_corrs[:] = np.nan
ML2_rly_corrs[:] = np.nan
ML3_sly_corrs[:] = np.nan
ML3_rly_corrs[:] = np.nan

ML1_sly_biass = np.zeros(len(refer_id_arr))
ML1_rly_biass = np.zeros(len(refer_id_arr))
ML2_sly_biass = np.zeros(len(refer_id_arr))
ML2_rly_biass = np.zeros(len(refer_id_arr))
ML3_sly_biass = np.zeros(len(refer_id_arr))
ML3_rly_biass = np.zeros(len(refer_id_arr))

ML1_sly_biass[:] = np.nan
ML1_rly_biass[:] = np.nan
ML2_sly_biass[:] = np.nan
ML2_rly_biass[:] = np.nan
ML3_sly_biass[:] = np.nan
ML3_rly_biass[:] = np.nan

ML1_sly_KGEs = np.zeros(len(refer_id_arr))
ML1_rly_KGEs = np.zeros(len(refer_id_arr))
ML2_sly_KGEs = np.zeros(len(refer_id_arr))
ML2_rly_KGEs = np.zeros(len(refer_id_arr))
ML3_sly_KGEs = np.zeros(len(refer_id_arr))
ML3_rly_KGEs = np.zeros(len(refer_id_arr))

ML1_sly_KGEs[:] = np.nan
ML1_rly_KGEs[:] = np.nan
ML2_sly_KGEs[:] = np.nan
ML2_rly_KGEs[:] = np.nan
ML3_sly_KGEs[:] = np.nan
ML3_rly_KGEs[:] = np.nan

ML1_sly_obs = np.array([])
ML1_rly_obs = np.array([])
ML2_sly_obs = np.array([])
ML2_rly_obs = np.array([])
ML3_sly_obs = np.array([])
ML3_rly_obs = np.array([])

ML1_sly_est = np.array([])
ML1_rly_est = np.array([])
ML2_sly_est = np.array([])
ML2_rly_est = np.array([])
ML3_sly_est = np.array([])
ML3_rly_est = np.array([])

for i in range(len(refer_id_arr)):
    idd = refer_id_arr[i]

    y_est = ML1_sly_y_pred_valid_arr[ML1_sly_y_pred_valid_arr[:, 0] == idd, 4]
    y_obs = ML1_sly_y_pred_valid_arr[ML1_sly_y_pred_valid_arr[:, 0] == idd, 8]
    ML1_sly_corrs[i], ML1_sly_biass[i], ML1_sly_RMSEs[i], ML1_sly_ubRMSEs[i], ML1_sly_KGEs[i] = accuracy(y_obs,
                                                                                                          y_est).values()
    ML1_sly_obs = np.append(ML1_sly_obs, y_obs.copy())
    ML1_sly_est = np.append(ML1_sly_est, y_est.copy())

    y_est = ML1_rly_y_pred_valid_arr[ML1_rly_y_pred_valid_arr[:, 0] == idd, 4]
    y_obs = ML1_rly_y_pred_valid_arr[ML1_rly_y_pred_valid_arr[:, 0] == idd, 8]
    ML1_rly_corrs[i], ML1_rly_biass[i], ML1_rly_RMSEs[i], ML1_rly_ubRMSEs[i], ML1_rly_KGEs[i] = accuracy(y_obs,
                                                                                                          y_est).values()
    ML1_rly_obs = np.append(ML1_rly_obs, y_obs.copy())
    ML1_rly_est = np.append(ML1_rly_est, y_est.copy())

    y_est = ML2_sly_y_pred_valid_arr[ML2_sly_y_pred_valid_arr[:, 0] == idd, 4]
    y_obs = ML2_sly_y_pred_valid_arr[ML2_sly_y_pred_valid_arr[:, 0] == idd, 8]
    ML2_sly_corrs[i], ML2_sly_biass[i], ML2_sly_RMSEs[i], ML2_sly_ubRMSEs[i], ML2_sly_KGEs[i] = accuracy(y_obs,
                                                                                                          y_est).values()
    ML2_sly_obs = np.append(ML2_sly_obs, y_obs.copy())
    ML2_sly_est = np.append(ML2_sly_est, y_est.copy())

    y_est = ML2_rly_y_pred_valid_arr[ML2_rly_y_pred_valid_arr[:, 0] == idd, 4]
    y_obs = ML2_rly_y_pred_valid_arr[ML2_rly_y_pred_valid_arr[:, 0] == idd, 8]
    ML2_rly_corrs[i], ML2_rly_biass[i], ML2_rly_RMSEs[i], ML2_rly_ubRMSEs[i], ML2_rly_KGEs[i] = accuracy(y_obs,
                                                                                                          y_est).values()
    ML2_rly_obs = np.append(ML2_rly_obs, y_obs.copy())
    ML2_rly_est = np.append(ML2_rly_est, y_est.copy())

    y_est = ML3_sly_y_pred_valid_arr[ML3_sly_y_pred_valid_arr[:, 0] == idd, 4]
    y_obs = ML3_sly_y_pred_valid_arr[ML3_sly_y_pred_valid_arr[:, 0] == idd, 8]
    ML3_sly_corrs[i], ML3_sly_biass[i], ML3_sly_RMSEs[i], ML3_sly_ubRMSEs[i], ML3_sly_KGEs[i] = accuracy(y_obs,
                                                                                                          y_est).values()
    ML3_sly_obs = np.append(ML3_sly_obs, y_obs.copy())
    ML3_sly_est = np.append(ML3_sly_est, y_est.copy())

    y_est = ML3_rly_y_pred_valid_arr[ML3_rly_y_pred_valid_arr[:, 0] == idd, 4]
    y_obs = ML3_rly_y_pred_valid_arr[ML3_rly_y_pred_valid_arr[:, 0] == idd, 8]
    ML3_rly_corrs[i], ML3_rly_biass[i], ML3_rly_RMSEs[i], ML3_rly_ubRMSEs[i], ML3_rly_KGEs[i] = accuracy(y_obs,
                                                                                                          y_est).values()
    ML3_rly_obs = np.append(ML3_rly_obs, y_obs.copy())
    ML3_rly_est = np.append(ML3_rly_est, y_est.copy())

df_ML_metrics = pd.DataFrame({
    'longitude': refer_lon_arr,
    'latitude': refer_lat_arr,
    'refer_id': refer_id_arr,
    'ML1_sly_corrs': ML1_sly_corrs,
    'ML1_sly_RMSEs': ML1_sly_RMSEs,
    'ML1_sly_biass': ML1_sly_biass,
    'ML1_sly_ubRMSEs': ML1_sly_ubRMSEs,
    'ML1_sly_KGEs': ML1_sly_KGEs,
    'ML1_rly_corrs': ML1_rly_corrs,
    'ML1_rly_RMSEs': ML1_rly_RMSEs,
    'ML1_rly_biass': ML1_rly_biass,
    'ML1_rly_ubRMSEs': ML1_rly_ubRMSEs,
    'ML1_rly_KGEs': ML1_rly_KGEs,
    'ML2_sly_corrs': ML2_sly_corrs,
    'ML2_sly_RMSEs': ML2_sly_RMSEs,
    'ML2_sly_biass': ML2_sly_biass,
    'ML2_sly_ubRMSEs': ML2_sly_ubRMSEs,
    'ML2_sly_KGEs': ML2_sly_KGEs,
    'ML2_rly_corrs': ML2_rly_corrs,
    'ML2_rly_RMSEs': ML2_rly_RMSEs,
    'ML2_rly_biass': ML2_rly_biass,
    'ML2_rly_ubRMSEs': ML2_rly_ubRMSEs,
    'ML2_rly_KGEs': ML2_rly_KGEs,
    'ML3_sly_corrs': ML3_sly_corrs,
    'ML3_sly_RMSEs': ML3_sly_RMSEs,
    'ML3_sly_biass': ML3_sly_biass,
    'ML3_sly_ubRMSEs': ML3_sly_ubRMSEs,
    'ML3_sly_KGEs': ML3_sly_KGEs,
    'ML3_rly_corrs': ML3_rly_corrs,
    'ML3_rly_RMSEs': ML3_rly_RMSEs,
    'ML3_rly_biass': ML3_rly_biass,
    'ML3_rly_ubRMSEs': ML3_rly_ubRMSEs,
    'ML3_rly_KGEs': ML3_rly_KGEs,
}, index=refer_id_arr)

ML1_sly_all_corr, ML1_sly_all_bias, ML1_sly_all_RMSE, ML1_sly_all_ubRMSE, ML1_sly_all_KGE = accuracy(ML1_sly_obs,
                                                                                                      ML1_sly_est).values()
ML1_rly_all_corr, ML1_rly_all_bias, ML1_rly_all_RMSE, ML1_rly_all_ubRMSE, ML1_rly_all_KGE = accuracy(ML1_rly_obs,
                                                                                                      ML1_rly_est).values()
ML2_sly_all_corr, ML2_sly_all_bias, ML2_sly_all_RMSE, ML2_sly_all_ubRMSE, ML2_sly_all_KGE = accuracy(ML2_sly_obs,
                                                                                                      ML2_sly_est).values()
ML2_rly_all_corr, ML2_rly_all_bias, ML2_rly_all_RMSE, ML2_rly_all_ubRMSE, ML2_rly_all_KGE = accuracy(ML2_rly_obs,
                                                                                                      ML2_rly_est).values()
ML3_sly_all_corr, ML3_sly_all_bias, ML3_sly_all_RMSE, ML3_sly_all_ubRMSE, ML3_sly_all_KGE = accuracy(ML3_sly_obs,
                                                                                                      ML3_sly_est).values()
ML3_rly_all_corr, ML3_rly_all_bias, ML3_rly_all_RMSE, ML3_rly_all_ubRMSE, ML3_rly_all_KGE = accuracy(ML3_rly_obs,
                                                                                                      ML3_rly_est).values()

print(
    f'ML1_sly: {ML1_sly_all_corr:.4f},{ML1_sly_all_bias:.4f},{ML1_sly_all_RMSE:.4f},{ML1_sly_all_ubRMSE:.4f},{ML1_sly_all_KGE:.4f}')
print(
    f'ML1_rly: {ML1_rly_all_corr:.4f},{ML1_rly_all_bias:.4f},{ML1_rly_all_RMSE:.4f},{ML1_rly_all_ubRMSE:.4f},{ML1_rly_all_KGE:.4f}')
print(
    f'ML2_sly: {ML2_sly_all_corr:.4f},{ML2_sly_all_bias:.4f},{ML2_sly_all_RMSE:.4f},{ML2_sly_all_ubRMSE:.4f},{ML2_sly_all_KGE:.4f}')
print(
    f'ML2_rly: {ML2_rly_all_corr:.4f},{ML2_rly_all_bias:.4f},{ML2_rly_all_RMSE:.4f},{ML2_rly_all_ubRMSE:.4f},{ML2_rly_all_KGE:.4f}')
print(
    f'ML3_sly: {ML3_sly_all_corr:.4f},{ML3_sly_all_bias:.4f},{ML3_sly_all_RMSE:.4f},{ML3_sly_all_ubRMSE:.4f},{ML3_sly_all_KGE:.4f}')
print(
    f'ML3_rly: {ML3_rly_all_corr:.4f},{ML3_rly_all_bias:.4f},{ML3_rly_all_RMSE:.4f},{ML3_rly_all_ubRMSE:.4f},{ML3_rly_all_KGE:.4f}')

df_ML_metrics.to_csv(f'{df_name}.csv')
print(df_ML_metrics.mean())
print(df_ML_metrics.median())

ML1_sly: 0.6155,0.0016,0.0853,0.0852,0.4646
ML1_rly: 0.2151,-0.0061,0.1176,0.1175,0.0424
ML2_sly: 0.5929,0.0007,0.0927,0.0927,0.5580
ML2_rly: 0.1073,0.0085,0.1445,0.1442,0.0837
ML3_sly: 0.7017,-0.0054,0.0794,0.0792,0.6424
ML3_rly: 0.1957,-0.0056,0.1185,0.1184,0.1250
longitude          67.716417
latitude           15.261463
refer_id           30.000000
ML1_sly_corrs       0.724170
ML1_sly_RMSEs       0.080256
ML1_sly_biass      -0.004340
ML1_sly_ubRMSEs     0.056419
ML1_sly_KGEs        0.457550
ML1_rly_corrs       0.447463
ML1_rly_RMSEs       0.111236
ML1_rly_biass       0.009500
ML1_rly_ubRMSEs     0.054604
ML1_rly_KGEs       -0.348638
ML2_sly_corrs       0.750119
ML2_sly_RMSEs       0.079974
ML2_sly_biass      -0.002562
ML2_sly_ubRMSEs     0.053232
ML2_sly_KGEs        0.494341
ML2_rly_corrs       0.470820
ML2_rly_RMSEs       0.126541
ML2_rly_biass       0.015276
ML2_rly_ubRMSEs     0.057505
ML2_rly_KGEs       -0.367661
ML3_sly_corrs       0.761253
ML3_sly_RMSEs       0.073977
ML3_sly_

## Train and save the model given optimal parameters

Given the optimal parameters, now, we can train the ML models.

In [12]:
# @title Load inputs and parameters
removed_indices_ML1 = [0, 1, 2, 3]
removed_indices_ML2 = [0, 1, 2, 3]
removed_indices_ML3 = [0, 1, 2, 3]


ML1_sly_input_valid = np.load('sentinel1_surface_X_valid.npy')
ML1_sly_y_valid = np.load('sentinel1_surface_y_valid.npy')
ML2_sly_input_valid = np.load('sentinel2_surface_X_valid.npy')
ML2_sly_y_valid = np.load('sentinel2_surface_y_valid.npy')
ML3_sly_input_valid = np.load('HLSL30_surface_X_valid.npy')
ML3_sly_y_valid = np.load('HLSL30_surface_y_valid.npy')
ML1_rly_input_valid = np.load('sentinel1_rootzone_X_valid.npy')
ML1_rly_y_valid = np.load('sentinel1_rootzone_y_valid.npy')
ML2_rly_input_valid = np.load('sentinel2_rootzone_X_valid.npy')
ML2_rly_y_valid = np.load('sentinel2_rootzone_y_valid.npy')
ML3_rly_input_valid = np.load('HLSL30_rootzone_X_valid.npy')
ML3_rly_y_valid = np.load('HLSL30_rootzone_y_valid.npy')

print(ML1_sly_input_valid.shape, ML1_sly_y_valid.shape)
print(ML2_sly_input_valid.shape, ML2_sly_y_valid.shape)
print(ML3_sly_input_valid.shape, ML3_sly_y_valid.shape)
print(ML1_rly_input_valid.shape, ML1_rly_y_valid.shape)
print(ML2_rly_input_valid.shape, ML2_rly_y_valid.shape)
print(ML3_rly_input_valid.shape, ML3_rly_y_valid.shape)

ML1_sly_input_info_valid = ML1_sly_input_valid[:, 0:4].copy()
ML2_sly_input_info_valid = ML2_sly_input_valid[:, 0:4].copy()
ML3_sly_input_info_valid = ML3_sly_input_valid[:, 0:4].copy()
ML1_rly_input_info_valid = ML1_rly_input_valid[:, 0:4].copy()
ML2_rly_input_info_valid = ML2_rly_input_valid[:, 0:4].copy()
ML3_rly_input_info_valid = ML3_rly_input_valid[:, 0:4].copy()

ML1_sly_input_valid = np.delete(ML1_sly_input_valid, removed_indices_ML1, axis=1)  #, 15
ML2_sly_input_valid = np.delete(ML2_sly_input_valid, removed_indices_ML2, axis=1)
ML3_sly_input_valid = np.delete(ML3_sly_input_valid, removed_indices_ML3, axis=1)
ML1_rly_input_valid = np.delete(ML1_rly_input_valid, removed_indices_ML1, axis=1)
ML2_rly_input_valid = np.delete(ML2_rly_input_valid, removed_indices_ML2, axis=1)
ML3_rly_input_valid = np.delete(ML3_rly_input_valid, removed_indices_ML3, axis=1)
# ML1_sly_input_all = np.delete(ML1_sly_input_all, [0, 1, 3, 15], axis=1)
# ML2_sly_input_all = np.delete(ML2_sly_input_all, [0, 1, 3, 15], axis=1)
# ML3_sly_input_all = np.delete(ML3_sly_input_all, [0, 1, 3, 15], axis=1)
# ML1_rly_input_all = np.delete(ML1_rly_input_all, [0, 1, 3, 15], axis=1)
# ML2_rly_input_all = np.delete(ML2_rly_input_all, [0, 1, 3, 15], axis=1)
# ML3_rly_input_all = np.delete(ML3_rly_input_all, [0, 1, 3, 15], axis=1)

print('after remove some variables: ')
print(ML1_sly_input_valid.shape, ML1_sly_y_valid.shape)
print(ML2_sly_input_valid.shape, ML2_sly_y_valid.shape)
print(ML3_sly_input_valid.shape, ML3_sly_y_valid.shape)
print(ML1_rly_input_valid.shape, ML1_rly_y_valid.shape)
print(ML2_rly_input_valid.shape, ML2_rly_y_valid.shape)
print(ML3_rly_input_valid.shape, ML3_rly_y_valid.shape)

kk = 0

with open('ML1_sly_lgb_reg_best_params.json','r') as f:
    ML1_sly_xgb_reg_best_params = json.load(f)
with open('ML1_rly_lgb_reg_best_params.json', 'r') as f:
    ML1_rly_xgb_reg_best_params = json.load(f)
with open('ML2_sly_lgb_reg_best_params.json', 'r') as f:
    ML2_sly_xgb_reg_best_params = json.load(f)
with open('ML2_rly_lgb_reg_best_params.json', 'r') as f:
    ML2_rly_xgb_reg_best_params = json.load(f)
with open('ML3_sly_lgb_reg_best_params.json', 'r') as f:
    ML3_sly_xgb_reg_best_params = json.load(f)
with open('ML3_rly_lgb_reg_best_params.json', 'r') as f:
    ML3_rly_xgb_reg_best_params = json.load(f)

np.set_printoptions(suppress=True, precision=6, floatmode='fixed')
print('sentinel1_surface: ')
print(np.nanmean(ML1_sly_input_valid, axis=0))
print('sentinel2_surface: ')
print(np.nanmean(ML2_sly_input_valid, axis=0))
print('HLSL30_surface: ')
print(np.nanmean(ML3_sly_input_valid, axis=0))
print('sentinel1_rootzone: ')
print(np.nanmean(ML1_rly_input_valid, axis=0))
print('sentinel2_rootzone:')
print(np.nanmean(ML2_rly_input_valid, axis=0))
print('HLSL30_rootzone: ')
print(np.nanmean(ML3_rly_input_valid, axis=0))



"""------------------------------ML1_sly------------------------------------------------"""
sample_weights = np.ones_like(ML1_sly_y_valid)
site_ids = ML1_sly_input_info_valid[:, 0].reshape(-1, 1)
for sid in np.unique(site_ids):
    q_low = np.quantile(ML1_sly_y_valid[site_ids == sid], 0.20)
    q_high = np.quantile(ML1_sly_y_valid[site_ids == sid], 0.80)
    sample_weights[(site_ids == sid) & (ML1_sly_y_valid < q_low)] = 2
    sample_weights[(site_ids == sid) & (ML1_sly_y_valid > q_high)] = 2
sample_weights = sample_weights.ravel()

id_arr = np.unique(ML1_sly_input_info_valid[:, 0])
print(id_arr.shape)
random.shuffle(id_arr)
id_arr_group = np.array_split(id_arr, 10)
site_ids_train = np.concatenate(id_arr_group[:8])
site_ids_calib = np.concatenate(id_arr_group[8:])
print('Train sites:', site_ids_train.shape)
print('Calib sites:', site_ids_calib.shape)

training_indice_valid = np.isin(ML1_sly_input_info_valid[:, 0], site_ids_train)
clib_indice_valid = np.isin(ML1_sly_input_info_valid[:, 0], site_ids_calib)
print(f'total: {len(training_indice_valid)}; training size: {np.count_nonzero(training_indice_valid)}; '
      f'clibration size: {np.count_nonzero(clib_indice_valid)}.')

X_train_valid = ML1_sly_input_valid[training_indice_valid, :]
y_train_valid = ML1_sly_y_valid[training_indice_valid]
X_calib_valid = ML1_sly_input_valid[clib_indice_valid, :]
y_calib_valid = ML1_sly_y_valid[clib_indice_valid]
sample_weights_train = sample_weights[training_indice_valid]

# standard scale
ML1_sly_scaler = StandardScaler()
ML1_sly_input_valid_scaled = ML1_sly_scaler.fit_transform(X_train_valid)
ML1_sly_input_calib_scaled = ML1_sly_scaler.transform(X_calib_valid)

# save scaler
dump(ML1_sly_scaler, 'ML1_sly_scaler.joblib')

# ML1_sly
best_params = ML1_sly_xgb_reg_best_params.copy()
ML1_sly_lgb_mean = lgb.LGBMRegressor(**best_params, verbosity=-1)
ML1_sly_lgb_mean.fit(ML1_sly_input_valid_scaled, y_train_valid.ravel(), sample_weight=sample_weights_train)

# Train LGBMRegressor models for _MapieQuantileRegressor
list_estimators_cqr = []
for alpha_ in [(1 - confidence_level) / 2, (1 + confidence_level) / 2, 0.5]:
    estimator_ = lgb.LGBMRegressor(**best_params,
                                    objective='quantile',
                                    alpha=alpha_, verbosity=-1
                                    )
    estimator_.fit(ML1_sly_input_valid_scaled, y_train_valid.ravel())
    list_estimators_cqr.append(estimator_)

# Conformalize uncertainties on conformalize set
ML1_sly_mapie_cqr = ConformalizedQuantileRegressor(
    list_estimators_cqr, confidence_level=0.9, prefit=True
)
ML1_sly_mapie_cqr.conformalize(ML1_sly_input_calib_scaled, y_calib_valid.ravel())

dump(ML1_sly_lgb_mean, 'ML1_sly_lgb_mean.joblib')
dump(ML1_sly_mapie_cqr, 'ML1_sly_mapie_cqr.joblib')

"""------------------------------ML1_rly------------------------------------------------"""
sample_weights = np.ones_like(ML1_rly_y_valid)
site_ids = ML1_rly_input_info_valid[:, 0].reshape(-1, 1)
for sid in np.unique(site_ids):
    q_low = np.quantile(ML1_rly_y_valid[site_ids == sid], 0.20)
    q_high = np.quantile(ML1_rly_y_valid[site_ids == sid], 0.80)
    sample_weights[(site_ids == sid) & (ML1_rly_y_valid < q_low)] = 2
    sample_weights[(site_ids == sid) & (ML1_rly_y_valid > q_high)] = 2
sample_weights = sample_weights.ravel()

id_arr = np.unique(ML1_rly_input_info_valid[:, 0])
print(id_arr.shape)
random.shuffle(id_arr)
id_arr_group = np.array_split(id_arr, 10)
site_ids_train = np.concatenate(id_arr_group[:8])
site_ids_calib = np.concatenate(id_arr_group[8:])
print('Train sites:', site_ids_train.shape)
print('Calib sites:', site_ids_calib.shape)

training_indice_valid = np.isin(ML1_rly_input_info_valid[:, 0], site_ids_train)
clib_indice_valid = np.isin(ML1_rly_input_info_valid[:, 0], site_ids_calib)
print(f'total: {len(training_indice_valid)}; training size: {np.count_nonzero(training_indice_valid)}; '
      f'clibration size: {np.count_nonzero(clib_indice_valid)}.')

X_train_valid = ML1_rly_input_valid[training_indice_valid, :]
y_train_valid = ML1_rly_y_valid[training_indice_valid]
X_calib_valid = ML1_rly_input_valid[clib_indice_valid, :]
y_calib_valid = ML1_rly_y_valid[clib_indice_valid]
sample_weights_train = sample_weights[training_indice_valid]

# standard scale
ML1_rly_scaler = StandardScaler()
ML1_rly_input_valid_scaled = ML1_rly_scaler.fit_transform(X_train_valid)
ML1_rly_input_calib_scaled = ML1_rly_scaler.transform(X_calib_valid)

# save scaler
dump(ML1_rly_scaler, 'ML1_rly_scaler.joblib')

# ML1_rly
best_params = ML1_rly_xgb_reg_best_params.copy()
ML1_rly_lgb_mean = lgb.LGBMRegressor(**best_params, verbosity=-1)
ML1_rly_lgb_mean.fit(ML1_rly_input_valid_scaled, y_train_valid.ravel(), sample_weight=sample_weights_train)

# Train LGBMRegressor models for _MapieQuantileRegressor
list_estimators_cqr = []
for alpha_ in [(1 - confidence_level) / 2, (1 + confidence_level) / 2, 0.5]:
    estimator_ = lgb.LGBMRegressor(**best_params,
                                    objective='quantile',
                                    alpha=alpha_, verbosity=-1
                                    )
    estimator_.fit(ML1_rly_input_valid_scaled, y_train_valid.ravel())
    list_estimators_cqr.append(estimator_)

# Conformalize uncertainties on conformalize set
ML1_rly_mapie_cqr = ConformalizedQuantileRegressor(
    list_estimators_cqr, confidence_level=0.9, prefit=True
)
ML1_rly_mapie_cqr.conformalize(ML1_rly_input_calib_scaled, y_calib_valid.ravel())

dump(ML1_rly_lgb_mean, 'ML1_rly_lgb_mean.joblib')
dump(ML1_rly_mapie_cqr, 'ML1_rly_mapie_cqr.joblib')

"""------------------------------ML2_sly------------------------------------------------"""
sample_weights = np.ones_like(ML2_sly_y_valid)
site_ids = ML2_sly_input_info_valid[:, 0].reshape(-1, 1)
for sid in np.unique(site_ids):
    q_low = np.quantile(ML2_sly_y_valid[site_ids == sid], 0.20)
    q_high = np.quantile(ML2_sly_y_valid[site_ids == sid], 0.80)
    sample_weights[(site_ids == sid) & (ML2_sly_y_valid < q_low)] = 2
    sample_weights[(site_ids == sid) & (ML2_sly_y_valid > q_high)] = 2
sample_weights = sample_weights.ravel()

id_arr = np.unique(ML2_sly_input_info_valid[:, 0])
print(id_arr.shape)
random.shuffle(id_arr)
id_arr_group = np.array_split(id_arr, 10)
site_ids_train = np.concatenate(id_arr_group[:8])
site_ids_calib = np.concatenate(id_arr_group[8:])
print('Train sites:', site_ids_train.shape)
print('Calib sites:', site_ids_calib.shape)

training_indice_valid = np.isin(ML2_sly_input_info_valid[:, 0], site_ids_train)
clib_indice_valid = np.isin(ML2_sly_input_info_valid[:, 0], site_ids_calib)
print(f'total: {len(training_indice_valid)}; training size: {np.count_nonzero(training_indice_valid)}; '
      f'clibration size: {np.count_nonzero(clib_indice_valid)}.')

X_train_valid = ML2_sly_input_valid[training_indice_valid, :]
y_train_valid = ML2_sly_y_valid[training_indice_valid]
X_calib_valid = ML2_sly_input_valid[clib_indice_valid, :]
y_calib_valid = ML2_sly_y_valid[clib_indice_valid]
sample_weights_train = sample_weights[training_indice_valid]

# standard scale
ML2_sly_scaler = StandardScaler()
ML2_sly_input_valid_scaled = ML2_sly_scaler.fit_transform(X_train_valid)
ML2_sly_input_calib_scaled = ML2_sly_scaler.transform(X_calib_valid)

# save scaler
dump(ML2_sly_scaler, 'ML2_sly_scaler.joblib')

# ML2_sly
best_params = ML2_sly_xgb_reg_best_params.copy()
ML2_sly_lgb_mean = lgb.LGBMRegressor(**best_params, verbosity=-1)
ML2_sly_lgb_mean.fit(ML2_sly_input_valid_scaled, y_train_valid.ravel(), sample_weight=sample_weights_train)

# Train LGBMRegressor models for _MapieQuantileRegressor
list_estimators_cqr = []
for alpha_ in [(1 - confidence_level) / 2, (1 + confidence_level) / 2, 0.5]:
    estimator_ = lgb.LGBMRegressor(**best_params,
                                    objective='quantile',
                                    alpha=alpha_, verbosity=-1
                                    )
    estimator_.fit(ML2_sly_input_valid_scaled, y_train_valid.ravel())
    list_estimators_cqr.append(estimator_)

# Conformalize uncertainties on conformalize set
ML2_sly_mapie_cqr = ConformalizedQuantileRegressor(
    list_estimators_cqr, confidence_level=0.9, prefit=True
)
ML2_sly_mapie_cqr.conformalize(ML2_sly_input_calib_scaled, y_calib_valid.ravel())

dump(ML2_sly_lgb_mean, 'ML2_sly_lgb_mean.joblib')
dump(ML2_sly_mapie_cqr, 'ML2_sly_mapie_cqr.joblib')

"""------------------------------ML2_rly------------------------------------------------"""
sample_weights = np.ones_like(ML2_rly_y_valid)
site_ids = ML2_rly_input_info_valid[:, 0].reshape(-1, 1)
for sid in np.unique(site_ids):
    q_low = np.quantile(ML2_rly_y_valid[site_ids == sid], 0.20)
    q_high = np.quantile(ML2_rly_y_valid[site_ids == sid], 0.80)
    sample_weights[(site_ids == sid) & (ML2_rly_y_valid < q_low)] = 2
    sample_weights[(site_ids == sid) & (ML2_rly_y_valid > q_high)] = 2
sample_weights = sample_weights.ravel()

id_arr = np.unique(ML2_rly_input_info_valid[:, 0])
print(id_arr.shape)
random.shuffle(id_arr)
id_arr_group = np.array_split(id_arr, 10)
site_ids_train = np.concatenate(id_arr_group[:8])
site_ids_calib = np.concatenate(id_arr_group[8:])
print('Train sites:', site_ids_train.shape)
print('Calib sites:', site_ids_calib.shape)

training_indice_valid = np.isin(ML2_rly_input_info_valid[:, 0], site_ids_train)
clib_indice_valid = np.isin(ML2_rly_input_info_valid[:, 0], site_ids_calib)
print(f'total: {len(training_indice_valid)}; training size: {np.count_nonzero(training_indice_valid)}; '
      f'clibration size: {np.count_nonzero(clib_indice_valid)}.')

X_train_valid = ML2_rly_input_valid[training_indice_valid, :]
y_train_valid = ML2_rly_y_valid[training_indice_valid]
X_calib_valid = ML2_rly_input_valid[clib_indice_valid, :]
y_calib_valid = ML2_rly_y_valid[clib_indice_valid]
sample_weights_train = sample_weights[training_indice_valid]

# standard scale
ML2_rly_scaler = StandardScaler()
ML2_rly_input_valid_scaled = ML2_rly_scaler.fit_transform(X_train_valid)
ML2_rly_input_calib_scaled = ML2_rly_scaler.transform(X_calib_valid)

# save scaler
dump(ML2_rly_scaler, 'ML2_rly_scaler.joblib')

# ML2_rly
best_params = ML2_rly_xgb_reg_best_params.copy()
ML2_rly_lgb_mean = lgb.LGBMRegressor(**best_params, verbosity=-1)
ML2_rly_lgb_mean.fit(ML2_rly_input_valid_scaled, y_train_valid.ravel(), sample_weight=sample_weights_train)

# Train LGBMRegressor models for _MapieQuantileRegressor
list_estimators_cqr = []
for alpha_ in [(1 - confidence_level) / 2, (1 + confidence_level) / 2, 0.5]:
    estimator_ = lgb.LGBMRegressor(**best_params,
                                    objective='quantile',
                                    alpha=alpha_, verbosity=-1
                                    )
    estimator_.fit(ML2_rly_input_valid_scaled, y_train_valid.ravel())
    list_estimators_cqr.append(estimator_)

# Conformalize uncertainties on conformalize set
ML2_rly_mapie_cqr = ConformalizedQuantileRegressor(
    list_estimators_cqr, confidence_level=0.9, prefit=True
)
ML2_rly_mapie_cqr.conformalize(ML2_rly_input_calib_scaled, y_calib_valid.ravel())

dump(ML2_rly_lgb_mean, 'ML2_rly_lgb_mean.joblib')
dump(ML2_rly_mapie_cqr, 'ML2_rly_mapie_cqr.joblib')

"""------------------------------ML3_sly------------------------------------------------"""
sample_weights = np.ones_like(ML3_sly_y_valid)
site_ids = ML3_sly_input_info_valid[:, 0].reshape(-1, 1)
for sid in np.unique(site_ids):
    q_low = np.quantile(ML3_sly_y_valid[site_ids == sid], 0.20)
    q_high = np.quantile(ML3_sly_y_valid[site_ids == sid], 0.80)
    sample_weights[(site_ids == sid) & (ML3_sly_y_valid < q_low)] = 2
    sample_weights[(site_ids == sid) & (ML3_sly_y_valid > q_high)] = 2
sample_weights = sample_weights.ravel()

id_arr = np.unique(ML3_sly_input_info_valid[:, 0])
print(id_arr.shape)
random.shuffle(id_arr)
id_arr_group = np.array_split(id_arr, 10)
site_ids_train = np.concatenate(id_arr_group[:8])
site_ids_calib = np.concatenate(id_arr_group[8:])
print('Train sites:', site_ids_train.shape)
print('Calib sites:', site_ids_calib.shape)

training_indice_valid = np.isin(ML3_sly_input_info_valid[:, 0], site_ids_train)
clib_indice_valid = np.isin(ML3_sly_input_info_valid[:, 0], site_ids_calib)
print(f'total: {len(training_indice_valid)}; training size: {np.count_nonzero(training_indice_valid)}; '
      f'clibration size: {np.count_nonzero(clib_indice_valid)}.')

X_train_valid = ML3_sly_input_valid[training_indice_valid, :]
y_train_valid = ML3_sly_y_valid[training_indice_valid]
X_calib_valid = ML3_sly_input_valid[clib_indice_valid, :]
y_calib_valid = ML3_sly_y_valid[clib_indice_valid]
sample_weights_train = sample_weights[training_indice_valid]

# standard scale
ML3_sly_scaler = StandardScaler()
ML3_sly_input_valid_scaled = ML3_sly_scaler.fit_transform(X_train_valid)
ML3_sly_input_calib_scaled = ML3_sly_scaler.transform(X_calib_valid)

# save scaler
dump(ML3_sly_scaler, 'ML3_sly_scaler.joblib')

# ML3_sly
best_params = ML3_sly_xgb_reg_best_params.copy()
ML3_sly_lgb_mean = lgb.LGBMRegressor(**best_params, verbosity=-1)
ML3_sly_lgb_mean.fit(ML3_sly_input_valid_scaled, y_train_valid.ravel(), sample_weight=sample_weights_train)

# Train LGBMRegressor models for _MapieQuantileRegressor
list_estimators_cqr = []
for alpha_ in [(1 - confidence_level) / 2, (1 + confidence_level) / 2, 0.5]:
    estimator_ = lgb.LGBMRegressor(**best_params,
                                    objective='quantile',
                                    alpha=alpha_, verbosity=-1
                                    )
    estimator_.fit(ML3_sly_input_valid_scaled, y_train_valid.ravel())
    list_estimators_cqr.append(estimator_)

# Conformalize uncertainties on conformalize set
ML3_sly_mapie_cqr = ConformalizedQuantileRegressor(
    list_estimators_cqr, confidence_level=0.9, prefit=True
)
ML3_sly_mapie_cqr.conformalize(ML3_sly_input_calib_scaled, y_calib_valid.ravel())

dump(ML3_sly_lgb_mean, 'ML3_sly_lgb_mean.joblib')
dump(ML3_sly_mapie_cqr, 'ML3_sly_mapie_cqr.joblib')

"""------------------------------ML3_rly------------------------------------------------"""
sample_weights = np.ones_like(ML3_rly_y_valid)
site_ids = ML3_rly_input_info_valid[:, 0].reshape(-1, 1)
for sid in np.unique(site_ids):
    q_low = np.quantile(ML3_rly_y_valid[site_ids == sid], 0.20)
    q_high = np.quantile(ML3_rly_y_valid[site_ids == sid], 0.80)
    sample_weights[(site_ids == sid) & (ML3_rly_y_valid < q_low)] = 2
    sample_weights[(site_ids == sid) & (ML3_rly_y_valid > q_high)] = 2
sample_weights = sample_weights.ravel()

id_arr = np.unique(ML3_rly_input_info_valid[:, 0])
print(id_arr.shape)
random.shuffle(id_arr)
id_arr_group = np.array_split(id_arr, 10)
site_ids_train = np.concatenate(id_arr_group[:8])
site_ids_calib = np.concatenate(id_arr_group[8:])
print('Train sites:', site_ids_train.shape)
print('Calib sites:', site_ids_calib.shape)

training_indice_valid = np.isin(ML3_rly_input_info_valid[:, 0], site_ids_train)
clib_indice_valid = np.isin(ML3_rly_input_info_valid[:, 0], site_ids_calib)
print(f'total: {len(training_indice_valid)}; training size: {np.count_nonzero(training_indice_valid)}; '
      f'clibration size: {np.count_nonzero(clib_indice_valid)}.')

X_train_valid = ML3_rly_input_valid[training_indice_valid, :]
y_train_valid = ML3_rly_y_valid[training_indice_valid]
X_calib_valid = ML3_rly_input_valid[clib_indice_valid, :]
y_calib_valid = ML3_rly_y_valid[clib_indice_valid]
sample_weights_train = sample_weights[training_indice_valid]

# standard scale
ML3_rly_scaler = StandardScaler()
ML3_rly_input_valid_scaled = ML3_rly_scaler.fit_transform(X_train_valid)
ML3_rly_input_calib_scaled = ML3_rly_scaler.transform(X_calib_valid)

# save scaler
dump(ML3_rly_scaler, 'ML3_rly_scaler.joblib')

# ML3_rly
best_params = ML3_rly_xgb_reg_best_params.copy()
ML3_rly_lgb_mean = lgb.LGBMRegressor(**best_params, verbosity=-1)
ML3_rly_lgb_mean.fit(ML3_rly_input_valid_scaled, y_train_valid.ravel(), sample_weight=sample_weights_train)

# Train LGBMRegressor models for _MapieQuantileRegressor
list_estimators_cqr = []
for alpha_ in [(1 - confidence_level) / 2, (1 + confidence_level) / 2, 0.5]:
    estimator_ = lgb.LGBMRegressor(**best_params,
                                    objective='quantile',
                                    alpha=alpha_, verbosity=-1
                                    )
    estimator_.fit(ML3_rly_input_valid_scaled, y_train_valid.ravel())
    list_estimators_cqr.append(estimator_)

# Conformalize uncertainties on conformalize set
ML3_rly_mapie_cqr = ConformalizedQuantileRegressor(
    list_estimators_cqr, confidence_level=0.9, prefit=True
)
ML3_rly_mapie_cqr.conformalize(ML3_rly_input_calib_scaled, y_calib_valid.ravel())

dump(ML3_rly_lgb_mean, 'ML3_rly_lgb_mean.joblib')
dump(ML3_rly_mapie_cqr, 'ML3_rly_mapie_cqr.joblib')


(114652, 32) (114652, 1)
(27096, 42) (27096, 1)
(6952, 37) (6952, 1)
(37025, 32) (37025, 1)
(7770, 42) (7770, 1)
(1921, 37) (1921, 1)
after remove some variables: 
(114652, 28) (114652, 1)
(27096, 38) (27096, 1)
(6952, 33) (6952, 1)
(37025, 28) (37025, 1)
(7770, 38) (7770, 1)
(1921, 33) (1921, 1)
sentinel1_surface: 
[    2.959997 11661.111947     8.177560    15.282551     0.962013
    12.748666     2.227925     0.501996     7.104992     6.580900
    15.306219    30.494703    -0.064447    -0.048491    19.367030
    32.555602   127.426744    40.610688   102.744859     1.652000
     2.109555   128.205867    -2.138352   -11.805042   -19.024358
    38.106594     0.622549     1.676694]
sentinel2_surface: 
[    3.710978 15488.916006     8.397075    17.085902     0.958980
    11.235493     0.953065     0.624860     8.688827     4.551909
    12.646907    27.309700    -0.047450    -0.122255    19.940816
    32.931798   127.565655    42.258758   146.345729     1.665346
     2.533869   134.561292 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

(20,)
Train sites: (16,)
Calib sites: (4,)
total: 37025; training size: 27278; clibration size: 9747.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

(61,)
Train sites: (49,)
Calib sites: (12,)
total: 27096; training size: 20694; clibration size: 6402.


...

(61,)
Train sites: (49,)
Calib sites: (12,)
total: 6952; training size: 5970; clibration size: 982.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

(20,)
Train sites: (16,)
Calib sites: (4,)
total: 1921; training size: 1638; clibration size: 283.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

['ML3_rly_mapie_cqr.joblib']